# Script-Reliance Intervention for Cross-Script Writer Verification

## Research Question

Does linearly accessible script information actually influence individual writer-verification decisions?

Previous experiments showed that batch-level alternating adversarial training consistently improved overall and cross-script writer verification across three training seeds, while script remained almost perfectly linearly accessible from the learned embeddings.

This creates an important distinction:

**Script leakage is not necessarily script reliance.**

A representation may contain information about script without every writer-verification decision depending on that information.

## Objective

This notebook performs a controlled representation intervention.

A linear script-sensitive direction is estimated using development-writer embeddings only. That direction is then removed from embeddings before recomputing writer-verification similarities on writer-disjoint validation pairs.

For each verification pair, the change between the original similarity and the script-intervened similarity will be measured.

The initial quantity of interest is:

\[
R_{\text{script}}(A,B)
=
\left|
s(A,B)
-
s^{-script}(A,B)
\right|
\]

where:

- \(s(A,B)\) is the original cosine similarity,
- \(s^{-script}(A,B)\) is the similarity after removing the estimated script-sensitive component.

At this stage, this quantity is interpreted as a **script-reliance diagnostic**, not yet as a validated uncertainty measure.

## Experimental Principles

- Fit the script direction using development writers only.
- Evaluate verification effects on writer-disjoint validation writers.
- Do not use the official test set for method design or selection.
- Preserve the existing preprocessing, embedding model, and verification protocol.
- Treat linear script removal as a controlled diagnostic intervention, not proof of complete script erasure.
- First establish whether script intervention changes verification decisions before testing whether script reliance predicts errors.

In [1]:
import gc
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from torch.utils.data import DataLoader
from torchvision.models import resnet18

from handwriting_cross_script_research.dataset import QUWIDataset

In [2]:
SPLIT_SEED = 42
PROBE_SEED = 42
EMBEDDING_BATCH_SIZE = 16

cwd = Path.cwd().resolve()

if (cwd / "pyproject.toml").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "pyproject.toml").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Could not locate project root."
    )

IMAGE_DIR = (
    Path.home()
    / "Documents"
    / "Handwriting"
    / "QUWI"
    / "extracted"
    / "images"
)

SPLIT_PATH = (
    PROJECT_ROOT
    / "splits"
    / "quwi_writer_disjoint_split_seed42.csv"
)

VALIDATION_PAIR_PATH = (
    PROJECT_ROOT
    / "splits"
    / "verification"
    / "quwi_validation_verification_pairs.csv"
)

NOTEBOOK16_CHECKPOINT_DIR = (
    PROJECT_ROOT
    / "checkpoints"
    / "script_adversarial_metric_learning"
)

NOTEBOOK17_CHECKPOINT_DIR = (
    PROJECT_ROOT
    / "checkpoints"
    / "multiseed_adversarial_robustness"
)

REPORT_DIR = (
    PROJECT_ROOT
    / "reports"
    / "script_reliance_intervention"
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

split_df = pd.read_csv(
    SPLIT_PATH
)

development_df = (
    split_df[
        split_df[
            "experiment_split"
        ] == "development_train"
    ]
    .reset_index(drop=True)
)

validation_df = (
    split_df[
        split_df[
            "experiment_split"
        ] == "validation"
    ]
    .reset_index(drop=True)
)

validation_pairs_df = pd.read_csv(
    VALIDATION_PAIR_PATH
)

checkpoint_candidates = []

for checkpoint_dir in [
    NOTEBOOK16_CHECKPOINT_DIR,
    NOTEBOOK17_CHECKPOINT_DIR,
]:
    if checkpoint_dir.exists():
        checkpoint_candidates.extend(
            sorted(
                checkpoint_dir.glob(
                    "*.pt"
                )
            )
        )

batch_candidates = [
    path
    for path in checkpoint_candidates
    if "batch" in path.name.lower()
]

print(
    "Project root:",
    PROJECT_ROOT,
)

print(
    "Development writers:",
    development_df[
        "writer"
    ].nunique(),
)

print(
    "Development images:",
    len(
        development_df
    ),
)

print(
    "Validation writers:",
    validation_df[
        "writer"
    ].nunique(),
)

print(
    "Validation images:",
    len(
        validation_df
    ),
)

print(
    "Validation pairs:",
    len(
        validation_pairs_df
    ),
)

print()

print(
    "Batch checkpoint candidates:"
)

for path in batch_candidates:
    print(
        "-",
        path.relative_to(
            PROJECT_ROOT
        ),
    )

Project root: /home/arijit/Documents/handwriting-cross-script-research
Development writers: 226
Development images: 904
Validation writers: 56
Validation images: 224
Validation pairs: 18816

Batch checkpoint candidates:
- checkpoints/multiseed_adversarial_robustness/seed123_batch_alt_lambda0p5_best.pt
- checkpoints/multiseed_adversarial_robustness/seed2026_batch_alt_lambda0p5_best.pt


In [3]:
IMAGENET_MEAN = torch.tensor(
    [0.485, 0.456, 0.406],
    dtype=torch.float32,
).view(
    1,
    3,
    1,
    1,
)

IMAGENET_STD = torch.tensor(
    [0.229, 0.224, 0.225],
    dtype=torch.float32,
).view(
    1,
    3,
    1,
    1,
)


class ScriptAdversarialWriterModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.backbone = resnet18(
            weights=None
        )

        self.backbone.fc = nn.Identity()

        self.script_classifier = nn.Linear(
            512,
            2,
        )

        self.register_buffer(
            "imagenet_mean",
            IMAGENET_MEAN,
        )

        self.register_buffer(
            "imagenet_std",
            IMAGENET_STD,
        )

    def encode(
        self,
        images,
    ):
        normalized_images = (
            images
            - self.imagenet_mean
        ) / self.imagenet_std

        embeddings = self.backbone(
            normalized_images
        )

        return F.normalize(
            embeddings,
            p=2,
            dim=1,
        )


TRAIN_DEVICE = (
    torch.device("cuda")
    if torch.cuda.is_available()
    else torch.device("cpu")
)

seed123_checkpoint_path = next(
    path
    for path in batch_candidates
    if path.name
    == "seed123_batch_alt_lambda0p5_best.pt"
)

seed123_checkpoint = torch.load(
    seed123_checkpoint_path,
    map_location="cpu",
    weights_only=False,
)

seed123_model = (
    ScriptAdversarialWriterModel()
    .to(
        TRAIN_DEVICE
    )
)

seed123_model.load_state_dict(
    seed123_checkpoint[
        "model_state_dict"
    ],
    strict=True,
)

seed123_model.eval()

print(
    "Device:",
    TRAIN_DEVICE,
)

print(
    "Checkpoint:",
    seed123_checkpoint_path.relative_to(
        PROJECT_ROOT
    ),
)

print(
    "Training seed:",
    seed123_checkpoint[
        "training_seed"
    ],
)

print(
    "Best epoch:",
    seed123_checkpoint[
        "epoch"
    ],
)

print(
    "Validation AUC:",
    seed123_checkpoint[
        "validation_auc"
    ],
)

print(
    "Validation EER (%):",
    100.0
    * seed123_checkpoint[
        "validation_eer"
    ],
)

Device: cuda
Checkpoint: checkpoints/multiseed_adversarial_robustness/seed123_batch_alt_lambda0p5_best.pt
Training seed: 123
Best epoch: 8
Validation AUC: 0.7764246418264277
Validation EER (%): 29.166666666666664


In [4]:
development_dataset = QUWIDataset(
    metadata=development_df,
    image_dir=IMAGE_DIR,
)

validation_dataset = QUWIDataset(
    metadata=validation_df,
    image_dir=IMAGE_DIR,
)

development_loader = DataLoader(
    development_dataset,
    batch_size=EMBEDDING_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=EMBEDDING_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
)


def extract_embeddings_and_labels(
    model,
    loader,
):
    model.eval()

    embeddings = []
    labels = []
    filenames = []

    with torch.no_grad():
        for batch in loader:
            images = batch[
                "image"
            ].to(
                TRAIN_DEVICE
            )

            batch_embeddings = (
                model.encode(
                    images
                )
                .detach()
                .cpu()
                .numpy()
            )

            batch_labels = (
                batch[
                    "language_label"
                ]
                .detach()
                .cpu()
                .numpy()
            )

            embeddings.append(
                batch_embeddings
            )

            labels.append(
                batch_labels
            )

            filenames.extend(
                list(
                    batch[
                        "filename"
                    ]
                )
            )

    return (
        np.concatenate(
            embeddings,
            axis=0,
        ),
        np.concatenate(
            labels,
            axis=0,
        ),
        filenames,
    )


development_embeddings, development_script_labels, development_filenames = (
    extract_embeddings_and_labels(
        seed123_model,
        development_loader,
    )
)

validation_embeddings, validation_script_labels, validation_filenames = (
    extract_embeddings_and_labels(
        seed123_model,
        validation_loader,
    )
)

print(
    "Development embedding shape:",
    development_embeddings.shape,
)

print(
    "Validation embedding shape:",
    validation_embeddings.shape,
)

print(
    "Development script counts:",
    np.bincount(
        development_script_labels
    ).tolist(),
)

print(
    "Validation script counts:",
    np.bincount(
        validation_script_labels
    ).tolist(),
)

print(
    "Mean development embedding norm:",
    np.linalg.norm(
        development_embeddings,
        axis=1,
    ).mean(),
)

print(
    "Mean validation embedding norm:",
    np.linalg.norm(
        validation_embeddings,
        axis=1,
    ).mean(),
)

Development embedding shape: (904, 512)
Validation embedding shape: (224, 512)
Development script counts: [452, 452]
Validation script counts: [112, 112]
Mean development embedding norm: 1.0
Mean validation embedding norm: 1.0


In [5]:
script_probe = Pipeline(
    [
        (
            "scaler",
            StandardScaler(),
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                random_state=PROBE_SEED,
            ),
        ),
    ]
)

script_probe.fit(
    development_embeddings,
    development_script_labels,
)

validation_script_probabilities = (
    script_probe.predict_proba(
        validation_embeddings
    )[
        :,
        1,
    ]
)

validation_script_predictions = (
    script_probe.predict(
        validation_embeddings
    )
)

validation_script_auc = roc_auc_score(
    validation_script_labels,
    validation_script_probabilities,
)

validation_script_accuracy = accuracy_score(
    validation_script_labels,
    validation_script_predictions,
)

scaler = script_probe.named_steps[
    "scaler"
]

classifier = script_probe.named_steps[
    "classifier"
]

standardized_script_direction = (
    classifier.coef_[
        0
    ]
)

original_script_direction = (
    standardized_script_direction
    / scaler.scale_
)

script_direction = (
    original_script_direction
    / np.linalg.norm(
        original_script_direction
    )
)

print(
    "Validation script accuracy:",
    validation_script_accuracy,
)

print(
    "Validation script AUC:",
    validation_script_auc,
)

print(
    "Original-space direction shape:",
    script_direction.shape,
)

print(
    "Original-space direction norm:",
    np.linalg.norm(
        script_direction
    ),
)

Validation script accuracy: 0.9955357142857143
Validation script AUC: 0.9992028061224489
Original-space direction shape: (512,)
Original-space direction norm: 1.0000000000000002


In [6]:
def remove_script_direction(
    embeddings,
    direction,
):
    projection_strength = (
        embeddings
        @ direction
    )

    residual_embeddings = (
        embeddings
        - np.outer(
            projection_strength,
            direction,
        )
    )

    residual_norms = np.linalg.norm(
        residual_embeddings,
        axis=1,
        keepdims=True,
    )

    residual_embeddings = (
        residual_embeddings
        / residual_norms
    )

    return (
        residual_embeddings,
        projection_strength,
    )


development_intervened_embeddings, development_projection_strength = (
    remove_script_direction(
        development_embeddings,
        script_direction,
    )
)

validation_intervened_embeddings, validation_projection_strength = (
    remove_script_direction(
        validation_embeddings,
        script_direction,
    )
)

original_probe_intervened_probabilities = (
    script_probe.predict_proba(
        validation_intervened_embeddings
    )[
        :,
        1,
    ]
)

original_probe_intervened_auc = (
    roc_auc_score(
        validation_script_labels,
        original_probe_intervened_probabilities,
    )
)

residual_script_probe = Pipeline(
    [
        (
            "scaler",
            StandardScaler(),
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                random_state=PROBE_SEED,
            ),
        ),
    ]
)

residual_script_probe.fit(
    development_intervened_embeddings,
    development_script_labels,
)

residual_validation_probabilities = (
    residual_script_probe.predict_proba(
        validation_intervened_embeddings
    )[
        :,
        1,
    ]
)

residual_validation_predictions = (
    residual_script_probe.predict(
        validation_intervened_embeddings
    )
)

residual_script_auc = roc_auc_score(
    validation_script_labels,
    residual_validation_probabilities,
)

residual_script_accuracy = accuracy_score(
    validation_script_labels,
    residual_validation_predictions,
)

remaining_direction_component = np.max(
    np.abs(
        validation_intervened_embeddings
        @ script_direction
    )
)

print(
    "Original probe AUC after intervention:",
    original_probe_intervened_auc,
)

print(
    "Fresh residual probe accuracy:",
    residual_script_accuracy,
)

print(
    "Fresh residual probe AUC:",
    residual_script_auc,
)

print(
    "Maximum remaining removed-direction component:",
    remaining_direction_component,
)

print(
    "Mean intervened embedding norm:",
    np.linalg.norm(
        validation_intervened_embeddings,
        axis=1,
    ).mean(),
)

Original probe AUC after intervention: 0.14512914540816327
Fresh residual probe accuracy: 0.9955357142857143
Fresh residual probe AUC: 0.9992028061224489
Maximum remaining removed-direction component: 5.095750210681871e-18
Mean intervened embedding norm: 1.0


In [7]:
original_intervened_logits = (
    script_probe.decision_function(
        validation_intervened_embeddings
    )
)

print(
    "Original probe logit minimum:",
    original_intervened_logits.min(),
)

print(
    "Original probe logit maximum:",
    original_intervened_logits.max(),
)

print(
    "Original probe logit range:",
    np.ptp(
        original_intervened_logits
    ),
)

print(
    "Original probe logit std:",
    original_intervened_logits.std(),
)

print(
    "Approximately constant:",
    np.allclose(
        original_intervened_logits,
        original_intervened_logits[
            0
        ],
        rtol=0.0,
        atol=1e-10,
    ),
)

Original probe logit minimum: -0.9174786521260474
Original probe logit maximum: -0.9174786521260252
Original probe logit range: 2.220446049250313e-14
Original probe logit std: 3.586365916558994e-15
Approximately constant: True


In [8]:
SCRIPT_REMOVAL_ROUNDS = 32


def fit_effective_script_direction(
    embeddings,
    labels,
):
    probe = Pipeline(
        [
            (
                "scaler",
                StandardScaler(),
            ),
            (
                "classifier",
                LogisticRegression(
                    max_iter=2000,
                    random_state=PROBE_SEED,
                ),
            ),
        ]
    )

    probe.fit(
        embeddings,
        labels,
    )

    scaler = probe.named_steps[
        "scaler"
    ]

    classifier = probe.named_steps[
        "classifier"
    ]

    direction = (
        classifier.coef_[
            0
        ]
        / scaler.scale_
    )

    return probe, direction


def orthogonalize_direction(
    direction,
    basis,
):
    residual_direction = (
        direction.copy()
    )

    for basis_direction in basis:
        residual_direction = (
            residual_direction
            - np.dot(
                residual_direction,
                basis_direction,
            )
            * basis_direction
        )

    norm = np.linalg.norm(
        residual_direction
    )

    return (
        residual_direction
        / norm
    )


iterative_development_embeddings = (
    development_embeddings.copy()
)

iterative_validation_embeddings = (
    validation_embeddings.copy()
)

script_basis = []
script_removal_rows = []

for round_index in range(
    1,
    SCRIPT_REMOVAL_ROUNDS + 1,
):
    _, raw_direction = (
        fit_effective_script_direction(
            iterative_development_embeddings,
            development_script_labels,
        )
    )

    new_direction = (
        orthogonalize_direction(
            raw_direction,
            script_basis,
        )
    )

    script_basis.append(
        new_direction
    )

    iterative_development_embeddings, _ = (
        remove_script_direction(
            iterative_development_embeddings,
            new_direction,
        )
    )

    iterative_validation_embeddings, _ = (
        remove_script_direction(
            iterative_validation_embeddings,
            new_direction,
        )
    )

    residual_probe = Pipeline(
        [
            (
                "scaler",
                StandardScaler(),
            ),
            (
                "classifier",
                LogisticRegression(
                    max_iter=2000,
                    random_state=PROBE_SEED,
                ),
            ),
        ]
    )

    residual_probe.fit(
        iterative_development_embeddings,
        development_script_labels,
    )

    development_probabilities = (
        residual_probe.predict_proba(
            iterative_development_embeddings
        )[
            :,
            1,
        ]
    )

    validation_probabilities = (
        residual_probe.predict_proba(
            iterative_validation_embeddings
        )[
            :,
            1,
        ]
    )

    development_auc = roc_auc_score(
        development_script_labels,
        development_probabilities,
    )

    validation_auc = roc_auc_score(
        validation_script_labels,
        validation_probabilities,
    )

    script_removal_rows.append(
        {
            "removed_dimensions": (
                round_index
            ),
            "development_script_auc": (
                development_auc
            ),
            "validation_script_auc": (
                validation_auc
            ),
        }
    )

    if round_index in [
        1,
        2,
        4,
        8,
        16,
        32,
    ]:
        print(
            f"Removed dimensions {round_index:02d} | "
            f"Development script AUC {development_auc:.6f} | "
            f"Validation script AUC {validation_auc:.6f}"
        )


script_removal_df = pd.DataFrame(
    script_removal_rows
)

script_basis_matrix = np.stack(
    script_basis,
    axis=0,
)

basis_gram_matrix = (
    script_basis_matrix
    @ script_basis_matrix.T
)

print()

print(
    "Script basis shape:",
    script_basis_matrix.shape,
)

print(
    "Maximum basis orthogonality error:",
    np.max(
        np.abs(
            basis_gram_matrix
            - np.eye(
                SCRIPT_REMOVAL_ROUNDS
            )
        )
    ),
)

Removed dimensions 01 | Development script AUC 1.000000 | Validation script AUC 0.999203
Removed dimensions 02 | Development script AUC 1.000000 | Validation script AUC 0.999362
Removed dimensions 04 | Development script AUC 1.000000 | Validation script AUC 0.999522
Removed dimensions 08 | Development script AUC 1.000000 | Validation script AUC 0.999442
Removed dimensions 16 | Development script AUC 1.000000 | Validation script AUC 0.999043
Removed dimensions 32 | Development script AUC 1.000000 | Validation script AUC 0.995695

Script basis shape: (32, 512)
Maximum basis orthogonality error: 9.992007221626409e-16


In [9]:
development_writer_map = dict(
    zip(
        development_df[
            "filename"
        ],
        development_df[
            "writer"
        ],
    )
)

development_writer_labels = np.array(
    [
        development_writer_map[
            filename
        ]
        for filename in development_filenames
    ]
)

development_writers = np.unique(
    development_writer_labels
)

inner_rng = np.random.default_rng(
    SPLIT_SEED
)

shuffled_writers = inner_rng.permutation(
    development_writers
)

monitor_writer_count = int(
    round(
        0.20
        * len(
            shuffled_writers
        )
    )
)

monitor_writers = shuffled_writers[
    :monitor_writer_count
]

subspace_train_writers = shuffled_writers[
    monitor_writer_count:
]

subspace_train_mask = np.isin(
    development_writer_labels,
    subspace_train_writers,
)

subspace_monitor_mask = np.isin(
    development_writer_labels,
    monitor_writers,
)

subspace_train_embeddings = (
    development_embeddings[
        subspace_train_mask
    ]
)

subspace_train_script_labels = (
    development_script_labels[
        subspace_train_mask
    ]
)

subspace_monitor_embeddings = (
    development_embeddings[
        subspace_monitor_mask
    ]
)

subspace_monitor_script_labels = (
    development_script_labels[
        subspace_monitor_mask
    ]
)

print(
    "Subspace-train writers:",
    len(
        subspace_train_writers
    ),
)

print(
    "Subspace-monitor writers:",
    len(
        monitor_writers
    ),
)

print(
    "Subspace-train images:",
    len(
        subspace_train_embeddings
    ),
)

print(
    "Subspace-monitor images:",
    len(
        subspace_monitor_embeddings
    ),
)

print(
    "Train script counts:",
    np.bincount(
        subspace_train_script_labels
    ).tolist(),
)

print(
    "Monitor script counts:",
    np.bincount(
        subspace_monitor_script_labels
    ).tolist(),
)

print(
    "Writer overlap:",
    len(
        set(
            subspace_train_writers
        ).intersection(
            set(
                monitor_writers
            )
        )
    ),
)

Subspace-train writers: 181
Subspace-monitor writers: 45
Subspace-train images: 724
Subspace-monitor images: 180
Train script counts: [362, 362]
Monitor script counts: [90, 90]
Writer overlap: 0


In [10]:
MAX_INNER_REMOVAL_DIMENSIONS = 128

inner_train_embeddings = (
    subspace_train_embeddings.copy()
)

inner_monitor_embeddings = (
    subspace_monitor_embeddings.copy()
)

inner_script_basis = []
inner_removal_rows = []

report_dimensions = {
    1,
    2,
    4,
    8,
    16,
    32,
    64,
    96,
    128,
}

for dimension in range(
    1,
    MAX_INNER_REMOVAL_DIMENSIONS + 1,
):
    _, raw_direction = (
        fit_effective_script_direction(
            inner_train_embeddings,
            subspace_train_script_labels,
        )
    )

    residual_direction = (
        raw_direction.copy()
    )

    for basis_direction in (
        inner_script_basis
    ):
        residual_direction = (
            residual_direction
            - np.dot(
                residual_direction,
                basis_direction,
            )
            * basis_direction
        )

    residual_direction_norm = (
        np.linalg.norm(
            residual_direction
        )
    )

    if residual_direction_norm < 1e-12:
        print(
            "Stopped at dimension:",
            dimension,
        )
        break

    new_direction = (
        residual_direction
        / residual_direction_norm
    )

    inner_script_basis.append(
        new_direction
    )

    inner_train_embeddings, _ = (
        remove_script_direction(
            inner_train_embeddings,
            new_direction,
        )
    )

    inner_monitor_embeddings, _ = (
        remove_script_direction(
            inner_monitor_embeddings,
            new_direction,
        )
    )

    residual_probe, _ = (
        fit_effective_script_direction(
            inner_train_embeddings,
            subspace_train_script_labels,
        )
    )

    train_probabilities = (
        residual_probe.predict_proba(
            inner_train_embeddings
        )[
            :,
            1
        ]
    )

    monitor_probabilities = (
        residual_probe.predict_proba(
            inner_monitor_embeddings
        )[
            :,
            1
        ]
    )

    train_auc = roc_auc_score(
        subspace_train_script_labels,
        train_probabilities,
    )

    monitor_auc = roc_auc_score(
        subspace_monitor_script_labels,
        monitor_probabilities,
    )

    inner_removal_rows.append(
        {
            "removed_dimensions": (
                dimension
            ),
            "train_script_auc": (
                train_auc
            ),
            "monitor_script_auc": (
                monitor_auc
            ),
        }
    )

    if dimension in report_dimensions:
        print(
            f"Removed dimensions {dimension:03d} | "
            f"Train script AUC {train_auc:.6f} | "
            f"Monitor script AUC {monitor_auc:.6f}"
        )


inner_removal_df = pd.DataFrame(
    inner_removal_rows
)

print()

print(
    "Minimum monitor script AUC:",
    inner_removal_df[
        "monitor_script_auc"
    ].min(),
)

print(
    "Dimension at minimum monitor AUC:",
    int(
        inner_removal_df.loc[
            inner_removal_df[
                "monitor_script_auc"
            ].idxmin(),
            "removed_dimensions",
        ]
    ),
)

Removed dimensions 001 | Train script AUC 1.000000 | Monitor script AUC 0.992099
Removed dimensions 002 | Train script AUC 1.000000 | Monitor script AUC 0.992346
Removed dimensions 004 | Train script AUC 1.000000 | Monitor script AUC 0.994815
Removed dimensions 008 | Train script AUC 1.000000 | Monitor script AUC 0.999753
Removed dimensions 016 | Train script AUC 1.000000 | Monitor script AUC 0.999877
Removed dimensions 032 | Train script AUC 1.000000 | Monitor script AUC 0.999506
Removed dimensions 064 | Train script AUC 0.999985 | Monitor script AUC 0.959630
Removed dimensions 096 | Train script AUC 0.971635 | Monitor script AUC 0.932963
Removed dimensions 128 | Train script AUC 0.827524 | Monitor script AUC 0.807284

Minimum monitor script AUC: 0.8035802469135803
Dimension at minimum monitor AUC: 126


In [11]:
CANDIDATE_REMOVAL_DIMENSIONS = [
    1,
    2,
    4,
    8,
    16,
    32,
    64,
    96,
    128,
]

candidate_dimension_df = (
    inner_removal_df[
        inner_removal_df[
            "removed_dimensions"
        ].isin(
            CANDIDATE_REMOVAL_DIMENSIONS
        )
    ]
    .copy()
    .reset_index(drop=True)
)

selected_row = candidate_dimension_df.loc[
    candidate_dimension_df[
        "monitor_script_auc"
    ].idxmin()
]

SELECTED_SCRIPT_SUBSPACE_DIMENSION = int(
    selected_row[
        "removed_dimensions"
    ]
)

print(
    candidate_dimension_df
    .round(6)
    .to_string(
        index=False
    )
)

print()

print(
    "Selected script-subspace dimension:",
    SELECTED_SCRIPT_SUBSPACE_DIMENSION,
)

print(
    "Development-monitor script AUC:",
    selected_row[
        "monitor_script_auc"
    ],
)

 removed_dimensions  train_script_auc  monitor_script_auc
                  1          1.000000            0.992099
                  2          1.000000            0.992346
                  4          1.000000            0.994815
                  8          1.000000            0.999753
                 16          1.000000            0.999877
                 32          1.000000            0.999506
                 64          0.999985            0.959630
                 96          0.971635            0.932963
                128          0.827524            0.807284

Selected script-subspace dimension: 128
Development-monitor script AUC: 0.8072839506172839


In [12]:
full_intervention_development_embeddings = (
    development_embeddings.copy()
)

full_intervention_validation_embeddings = (
    validation_embeddings.copy()
)

final_script_basis = []

for dimension in range(
    1,
    SELECTED_SCRIPT_SUBSPACE_DIMENSION + 1,
):
    _, raw_direction = (
        fit_effective_script_direction(
            full_intervention_development_embeddings,
            development_script_labels,
        )
    )

    residual_direction = (
        raw_direction.copy()
    )

    for basis_direction in (
        final_script_basis
    ):
        residual_direction = (
            residual_direction
            - np.dot(
                residual_direction,
                basis_direction,
            )
            * basis_direction
        )

    residual_direction_norm = np.linalg.norm(
        residual_direction
    )

    if residual_direction_norm < 1e-12:
        raise RuntimeError(
            f"Degenerate direction at dimension {dimension}."
        )

    new_direction = (
        residual_direction
        / residual_direction_norm
    )

    final_script_basis.append(
        new_direction
    )

    full_intervention_development_embeddings, _ = (
        remove_script_direction(
            full_intervention_development_embeddings,
            new_direction,
        )
    )

    full_intervention_validation_embeddings, _ = (
        remove_script_direction(
            full_intervention_validation_embeddings,
            new_direction,
        )
    )


final_script_basis_matrix = np.stack(
    final_script_basis,
    axis=0,
)

final_residual_probe = Pipeline(
    [
        (
            "scaler",
            StandardScaler(),
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                random_state=PROBE_SEED,
            ),
        ),
    ]
)

final_residual_probe.fit(
    full_intervention_development_embeddings,
    development_script_labels,
)

final_validation_script_probabilities = (
    final_residual_probe.predict_proba(
        full_intervention_validation_embeddings
    )[
        :,
        1,
    ]
)

final_validation_script_predictions = (
    final_residual_probe.predict(
        full_intervention_validation_embeddings
    )
)

final_validation_script_auc = roc_auc_score(
    validation_script_labels,
    final_validation_script_probabilities,
)

final_validation_script_accuracy = accuracy_score(
    validation_script_labels,
    final_validation_script_predictions,
)

final_basis_gram = (
    final_script_basis_matrix
    @ final_script_basis_matrix.T
)

maximum_removed_component = np.max(
    np.abs(
        full_intervention_validation_embeddings
        @ final_script_basis_matrix.T
    )
)

print(
    "Final script basis shape:",
    final_script_basis_matrix.shape,
)

print(
    "Residual validation script accuracy:",
    final_validation_script_accuracy,
)

print(
    "Residual validation script AUC:",
    final_validation_script_auc,
)

print(
    "Maximum basis orthogonality error:",
    np.max(
        np.abs(
            final_basis_gram
            - np.eye(
                SELECTED_SCRIPT_SUBSPACE_DIMENSION
            )
        )
    ),
)

print(
    "Maximum remaining removed-subspace component:",
    maximum_removed_component,
)

print(
    "Mean intervened embedding norm:",
    np.linalg.norm(
        full_intervention_validation_embeddings,
        axis=1,
    ).mean(),
)

Final script basis shape: (128, 512)
Residual validation script accuracy: 0.7633928571428571
Residual validation script AUC: 0.8254145408163265
Maximum basis orthogonality error: 2.7917121739329254e-15
Maximum remaining removed-subspace component: 2.7749070402594e-15
Mean intervened embedding norm: 1.0


In [13]:
def calculate_interpolated_eer(
    labels,
    scores,
):
    fpr, tpr, thresholds = roc_curve(
        labels,
        scores,
    )

    fnr = 1.0 - tpr
    difference = fpr - fnr

    crossing_indices = np.where(
        np.diff(
            np.sign(
                difference
            )
        ) != 0
    )[0]

    if len(
        crossing_indices
    ) == 0:
        nearest_index = np.argmin(
            np.abs(
                difference
            )
        )

        eer = (
            fpr[
                nearest_index
            ]
            + fnr[
                nearest_index
            ]
        ) / 2.0

        threshold = thresholds[
            nearest_index
        ]

        return float(
            eer
        ), float(
            threshold
        )

    index = crossing_indices[
        0
    ]

    x0 = difference[
        index
    ]

    x1 = difference[
        index
        + 1
    ]

    weight = (
        -x0
        / (
            x1
            - x0
        )
    )

    eer = (
        fpr[
            index
        ]
        + weight
        * (
            fpr[
                index
                + 1
            ]
            - fpr[
                index
            ]
        )
    )

    threshold = (
        thresholds[
            index
        ]
        + weight
        * (
            thresholds[
                index
                + 1
            ]
            - thresholds[
                index
            ]
        )
    )

    return float(
        eer
    ), float(
        threshold
    )


original_embedding_map = dict(
    zip(
        validation_filenames,
        validation_embeddings,
    )
)

intervened_embedding_map = dict(
    zip(
        validation_filenames,
        full_intervention_validation_embeddings,
    )
)

original_embeddings_a = np.stack(
    [
        original_embedding_map[
            filename
        ]
        for filename in (
            validation_pairs_df[
                "filename_a"
            ]
        )
    ]
)

original_embeddings_b = np.stack(
    [
        original_embedding_map[
            filename
        ]
        for filename in (
            validation_pairs_df[
                "filename_b"
            ]
        )
    ]
)

intervened_embeddings_a = np.stack(
    [
        intervened_embedding_map[
            filename
        ]
        for filename in (
            validation_pairs_df[
                "filename_a"
            ]
        )
    ]
)

intervened_embeddings_b = np.stack(
    [
        intervened_embedding_map[
            filename
        ]
        for filename in (
            validation_pairs_df[
                "filename_b"
            ]
        )
    ]
)

pair_labels = validation_pairs_df[
    "pair_label"
].to_numpy(
    dtype=np.int64
)

original_scores = np.sum(
    original_embeddings_a
    * original_embeddings_b,
    axis=1,
)

intervened_scores = np.sum(
    intervened_embeddings_a
    * intervened_embeddings_b,
    axis=1,
)

score_shift = (
    intervened_scores
    - original_scores
)

script_reliance = np.abs(
    score_shift
)

pair_effect_df = (
    validation_pairs_df
    .copy()
    .reset_index(drop=True)
)

pair_effect_df[
    "original_score"
] = original_scores

pair_effect_df[
    "intervened_score"
] = intervened_scores

pair_effect_df[
    "score_shift"
] = score_shift

pair_effect_df[
    "script_reliance"
] = script_reliance

original_auc = roc_auc_score(
    pair_labels,
    original_scores,
)

intervened_auc = roc_auc_score(
    pair_labels,
    intervened_scores,
)

original_eer, original_eer_threshold = (
    calculate_interpolated_eer(
        pair_labels,
        original_scores,
    )
)

intervened_eer, intervened_eer_threshold = (
    calculate_interpolated_eer(
        pair_labels,
        intervened_scores,
    )
)

print(
    "Original verification AUC:",
    original_auc,
)

print(
    "Checkpoint verification AUC:",
    seed123_checkpoint[
        "validation_auc"
    ],
)

print(
    "Original verification EER (%):",
    100.0
    * original_eer,
)

print(
    "Intervened verification AUC:",
    intervened_auc,
)

print(
    "Intervened verification EER (%):",
    100.0
    * intervened_eer,
)

print()

print(
    "Mean script reliance:",
    script_reliance.mean(),
)

print(
    "Median script reliance:",
    np.median(
        script_reliance
    ),
)

print(
    "95th percentile script reliance:",
    np.quantile(
        script_reliance,
        0.95,
    ),
)

print(
    "Maximum script reliance:",
    script_reliance.max(),
)

Original verification AUC: 0.7764246418264277
Checkpoint verification AUC: 0.7764246418264277
Original verification EER (%): 29.166666666666664
Intervened verification AUC: 0.7765837585034013
Intervened verification EER (%): 29.664502164502167

Mean script reliance: 0.009797601673933388
Median script reliance: 0.006948152224113824
95th percentile script reliance: 0.029469532672088877
Maximum script reliance: 0.08265791215938512


In [14]:
CONDITION_BY_PAGE_PAIR = {
    (1, 2): "arabic_variable_same",
    (3, 4): "english_variable_same",
    (1, 3): "cross_variable_variable",
    (1, 4): "cross_variable_same",
    (2, 3): "cross_same_variable",
    (2, 4): "cross_same_same",
}


def filename_page_id(
    filename,
):
    return int(
        Path(
            filename
        ).stem.split(
            "_"
        )[-1]
    )


pair_effect_df[
    "page_a"
] = pair_effect_df[
    "filename_a"
].map(
    filename_page_id
)

pair_effect_df[
    "page_b"
] = pair_effect_df[
    "filename_b"
].map(
    filename_page_id
)

pair_effect_df[
    "condition"
] = [
    CONDITION_BY_PAGE_PAIR[
        (
            int(
                page_a
            ),
            int(
                page_b
            ),
        )
    ]
    for page_a, page_b in zip(
        pair_effect_df[
            "page_a"
        ],
        pair_effect_df[
            "page_b"
        ],
    )
]

pair_effect_df[
    "family"
] = np.where(
    pair_effect_df[
        "condition"
    ].str.startswith(
        "cross_"
    ),
    "cross_script",
    "within_script",
)

condition_rows = []

for condition, condition_df in (
    pair_effect_df.groupby(
        "condition",
        sort=False,
    )
):
    labels = condition_df[
        "pair_label"
    ].to_numpy(
        dtype=np.int64
    )

    condition_original_auc = (
        roc_auc_score(
            labels,
            condition_df[
                "original_score"
            ],
        )
    )

    condition_intervened_auc = (
        roc_auc_score(
            labels,
            condition_df[
                "intervened_score"
            ],
        )
    )

    condition_rows.append(
        {
            "condition": (
                condition
            ),
            "original_auc": (
                condition_original_auc
            ),
            "intervened_auc": (
                condition_intervened_auc
            ),
            "auc_change": (
                condition_intervened_auc
                - condition_original_auc
            ),
            "mean_script_reliance": (
                condition_df[
                    "script_reliance"
                ].mean()
            ),
            "median_script_reliance": (
                condition_df[
                    "script_reliance"
                ].median()
            ),
            "p95_script_reliance": (
                condition_df[
                    "script_reliance"
                ].quantile(
                    0.95
                )
            ),
            "mean_score_shift": (
                condition_df[
                    "score_shift"
                ].mean()
            ),
        }
    )


condition_effect_df = pd.DataFrame(
    condition_rows
)

family_reliance_df = (
    pair_effect_df
    .groupby(
        "family"
    )
    .agg(
        mean_script_reliance=(
            "script_reliance",
            "mean",
        ),
        median_script_reliance=(
            "script_reliance",
            "median",
        ),
        p95_script_reliance=(
            "script_reliance",
            lambda values: values.quantile(
                0.95
            ),
        ),
        mean_score_shift=(
            "score_shift",
            "mean",
        ),
    )
    .reset_index()
)

label_reliance_df = (
    pair_effect_df
    .groupby(
        "pair_label"
    )
    .agg(
        mean_script_reliance=(
            "script_reliance",
            "mean",
        ),
        median_script_reliance=(
            "script_reliance",
            "median",
        ),
        mean_score_shift=(
            "score_shift",
            "mean",
        ),
    )
    .reset_index()
)

print(
    condition_effect_df
    .round(6)
    .to_string(
        index=False
    )
)

print()

print(
    family_reliance_df
    .round(6)
    .to_string(
        index=False
    )
)

print()

print(
    label_reliance_df
    .round(6)
    .to_string(
        index=False
    )
)

              condition  original_auc  intervened_auc  auc_change  mean_script_reliance  median_script_reliance  p95_script_reliance  mean_score_shift
   arabic_variable_same      0.859740        0.859920    0.000180              0.003884                0.003047             0.010140         -0.003153
  english_variable_same      0.828374        0.825110   -0.003264              0.019352                0.018163             0.038892         -0.019191
cross_variable_variable      0.753305        0.753026   -0.000278              0.007786                0.005938             0.021392          0.001616
    cross_variable_same      0.710059        0.710662    0.000603              0.009411                0.007141             0.025796          0.004123
    cross_same_variable      0.758517        0.757201   -0.001316              0.007650                0.005713             0.020801          0.003054
        cross_same_same      0.768263        0.765631   -0.002632              0.010703       

In [15]:
decision_threshold = float(
    seed123_checkpoint[
        "validation_eer_threshold"
    ]
)

assert np.isclose(
    decision_threshold,
    original_eer_threshold,
    rtol=0.0,
    atol=1e-12,
)

pair_effect_df[
    "original_prediction"
] = (
    pair_effect_df[
        "original_score"
    ]
    >= decision_threshold
).astype(
    np.int64
)

pair_effect_df[
    "intervened_prediction"
] = (
    pair_effect_df[
        "intervened_score"
    ]
    >= decision_threshold
).astype(
    np.int64
)

pair_effect_df[
    "original_error"
] = (
    pair_effect_df[
        "original_prediction"
    ]
    != pair_effect_df[
        "pair_label"
    ]
).astype(
    np.int64
)

pair_effect_df[
    "intervened_error"
] = (
    pair_effect_df[
        "intervened_prediction"
    ]
    != pair_effect_df[
        "pair_label"
    ]
).astype(
    np.int64
)

pair_effect_df[
    "decision_flip"
] = (
    pair_effect_df[
        "original_prediction"
    ]
    != pair_effect_df[
        "intervened_prediction"
    ]
)

pair_effect_df[
    "margin_uncertainty"
] = (
    -np.abs(
        pair_effect_df[
            "original_score"
        ]
        - decision_threshold
    )
)

script_reliance_error_auc = roc_auc_score(
    pair_effect_df[
        "original_error"
    ],
    pair_effect_df[
        "script_reliance"
    ],
)

margin_error_auc = roc_auc_score(
    pair_effect_df[
        "original_error"
    ],
    pair_effect_df[
        "margin_uncertainty"
    ],
)

correct_reliance = pair_effect_df.loc[
    pair_effect_df[
        "original_error"
    ] == 0,
    "script_reliance",
]

error_reliance = pair_effect_df.loc[
    pair_effect_df[
        "original_error"
    ] == 1,
    "script_reliance",
]

rescued = (
    (
        pair_effect_df[
            "original_error"
        ] == 1
    )
    & (
        pair_effect_df[
            "intervened_error"
        ] == 0
    )
).sum()

harmed = (
    (
        pair_effect_df[
            "original_error"
        ] == 0
    )
    & (
        pair_effect_df[
            "intervened_error"
        ] == 1
    )
).sum()

print(
    "Decision threshold:",
    decision_threshold,
)

print(
    "Original errors:",
    int(
        pair_effect_df[
            "original_error"
        ].sum()
    ),
)

print(
    "Original error rate:",
    pair_effect_df[
        "original_error"
    ].mean(),
)

print()

print(
    "Correct-pair mean script reliance:",
    correct_reliance.mean(),
)

print(
    "Error-pair mean script reliance:",
    error_reliance.mean(),
)

print(
    "Correct-pair median script reliance:",
    correct_reliance.median(),
)

print(
    "Error-pair median script reliance:",
    error_reliance.median(),
)

print()

print(
    "Script-reliance error-detection AUC:",
    script_reliance_error_auc,
)

print(
    "Margin-uncertainty error-detection AUC:",
    margin_error_auc,
)

print()

print(
    "Decision flips:",
    int(
        pair_effect_df[
            "decision_flip"
        ].sum()
    ),
)

print(
    "Errors rescued by intervention:",
    int(
        rescued
    ),
)

print(
    "Correct decisions harmed by intervention:",
    int(
        harmed
    ),
)

Decision threshold: 0.6933626851131176
Original errors: 5485
Original error rate: 0.29150722789115646

Correct-pair mean script reliance: 0.009886793390798107
Error-pair mean script reliance: 0.009580825962443222
Correct-pair median script reliance: 0.006870147362525003
Error-pair median script reliance: 0.007188690471245818

Script-reliance error-detection AUC: 0.4843220033879675
Margin-uncertainty error-detection AUC: 0.7527732804471412

Decision flips: 466
Errors rescued by intervention: 141
Correct decisions harmed by intervention: 325


In [16]:
def summarize_uncertainty_by_group(
    dataframe,
    group_column,
):
    rows = []

    for group_value, group_df in (
        dataframe.groupby(
            group_column,
            sort=False,
        )
    ):
        error_labels = group_df[
            "original_error"
        ].to_numpy(
            dtype=np.int64
        )

        if np.unique(
            error_labels
        ).size == 2:
            reliance_auc = roc_auc_score(
                error_labels,
                group_df[
                    "script_reliance"
                ],
            )

            margin_auc = roc_auc_score(
                error_labels,
                group_df[
                    "margin_uncertainty"
                ],
            )
        else:
            reliance_auc = np.nan
            margin_auc = np.nan

        correct_mask = (
            group_df[
                "original_error"
            ] == 0
        )

        error_mask = (
            group_df[
                "original_error"
            ] == 1
        )

        rows.append(
            {
                group_column: (
                    group_value
                ),
                "pairs": len(
                    group_df
                ),
                "error_rate": (
                    group_df[
                        "original_error"
                    ].mean()
                ),
                "correct_mean_reliance": (
                    group_df.loc[
                        correct_mask,
                        "script_reliance",
                    ].mean()
                ),
                "error_mean_reliance": (
                    group_df.loc[
                        error_mask,
                        "script_reliance",
                    ].mean()
                ),
                "script_reliance_error_auc": (
                    reliance_auc
                ),
                "margin_error_auc": (
                    margin_auc
                ),
                "decision_flips": int(
                    group_df[
                        "decision_flip"
                    ].sum()
                ),
            }
        )

    return pd.DataFrame(
        rows
    )


family_uncertainty_df = (
    summarize_uncertainty_by_group(
        pair_effect_df,
        "family",
    )
)

condition_uncertainty_df = (
    summarize_uncertainty_by_group(
        pair_effect_df,
        "condition",
    )
)

print(
    family_uncertainty_df
    .round(6)
    .to_string(
        index=False
    )
)

print()

print(
    condition_uncertainty_df
    .round(6)
    .to_string(
        index=False
    )
)

       family  pairs  error_rate  correct_mean_reliance  error_mean_reliance  script_reliance_error_auc  margin_error_auc  decision_flips
within_script   6272    0.311224               0.013903             0.006561                   0.288785          0.744599             130
 cross_script  12544    0.281649               0.007961             0.011249                   0.597304          0.757204             336

              condition  pairs  error_rate  correct_mean_reliance  error_mean_reliance  script_reliance_error_auc  margin_error_auc  decision_flips
   arabic_variable_same   3136    0.319834               0.004918             0.001684                   0.176370          0.758779              12
  english_variable_same   3136    0.302615               0.022667             0.011714                   0.193143          0.736094             118
cross_variable_variable   3136    0.268176               0.007351             0.008972                   0.537779          0.777917          

In [17]:
validation_writer_map = dict(
    zip(
        validation_df[
            "filename"
        ],
        validation_df[
            "writer"
        ],
    )
)

pair_effect_df[
    "writer_a"
] = pair_effect_df[
    "filename_a"
].map(
    validation_writer_map
)

pair_effect_df[
    "writer_b"
] = pair_effect_df[
    "filename_b"
].map(
    validation_writer_map
)

validation_writers = np.sort(
    validation_df[
        "writer"
    ].unique()
)

inner_validation_rng = (
    np.random.default_rng(
        SPLIT_SEED
    )
)

shuffled_validation_writers = (
    inner_validation_rng.permutation(
        validation_writers
    )
)

split_point = len(
    shuffled_validation_writers
) // 2

uncertainty_calibration_writers = (
    shuffled_validation_writers[
        :split_point
    ]
)

uncertainty_evaluation_writers = (
    shuffled_validation_writers[
        split_point:
    ]
)

cross_script_pairs_df = (
    pair_effect_df[
        pair_effect_df[
            "family"
        ] == "cross_script"
    ]
    .copy()
    .reset_index(drop=True)
)

calibration_pair_mask = (
    cross_script_pairs_df[
        "writer_a"
    ].isin(
        uncertainty_calibration_writers
    )
    & cross_script_pairs_df[
        "writer_b"
    ].isin(
        uncertainty_calibration_writers
    )
)

evaluation_pair_mask = (
    cross_script_pairs_df[
        "writer_a"
    ].isin(
        uncertainty_evaluation_writers
    )
    & cross_script_pairs_df[
        "writer_b"
    ].isin(
        uncertainty_evaluation_writers
    )
)

uncertainty_calibration_df = (
    cross_script_pairs_df[
        calibration_pair_mask
    ]
    .copy()
    .reset_index(drop=True)
)

uncertainty_evaluation_df = (
    cross_script_pairs_df[
        evaluation_pair_mask
    ]
    .copy()
    .reset_index(drop=True)
)

calibration_eer, calibration_threshold = (
    calculate_interpolated_eer(
        uncertainty_calibration_df[
            "pair_label"
        ].to_numpy(
            dtype=np.int64
        ),
        uncertainty_calibration_df[
            "original_score"
        ].to_numpy(),
    )
)

for dataframe in [
    uncertainty_calibration_df,
    uncertainty_evaluation_df,
]:
    dataframe[
        "prediction"
    ] = (
        dataframe[
            "original_score"
        ]
        >= calibration_threshold
    ).astype(
        np.int64
    )

    dataframe[
        "verification_error"
    ] = (
        dataframe[
            "prediction"
        ]
        != dataframe[
            "pair_label"
        ]
    ).astype(
        np.int64
    )

    dataframe[
        "margin_uncertainty"
    ] = (
        -np.abs(
            dataframe[
                "original_score"
            ]
            - calibration_threshold
        )
    )


print(
    "Calibration writers:",
    len(
        uncertainty_calibration_writers
    ),
)

print(
    "Evaluation writers:",
    len(
        uncertainty_evaluation_writers
    ),
)

print(
    "Writer overlap:",
    len(
        set(
            uncertainty_calibration_writers
        ).intersection(
            set(
                uncertainty_evaluation_writers
            )
        )
    ),
)

print()

print(
    "Calibration cross-script pairs:",
    len(
        uncertainty_calibration_df
    ),
)

print(
    "Evaluation cross-script pairs:",
    len(
        uncertainty_evaluation_df
    ),
)

print(
    "Calibration genuine/impostor:",
    uncertainty_calibration_df[
        "pair_label"
    ].value_counts().sort_index().to_dict(),
)

print(
    "Evaluation genuine/impostor:",
    uncertainty_evaluation_df[
        "pair_label"
    ].value_counts().sort_index().to_dict(),
)

print()

print(
    "Calibration-derived threshold:",
    calibration_threshold,
)

print(
    "Calibration EER (%):",
    100.0
    * calibration_eer,
)

print(
    "Calibration error rate:",
    uncertainty_calibration_df[
        "verification_error"
    ].mean(),
)

print(
    "Evaluation error rate:",
    uncertainty_evaluation_df[
        "verification_error"
    ].mean(),
)

Calibration writers: 28
Evaluation writers: 28
Writer overlap: 0

Calibration cross-script pairs: 3136
Evaluation cross-script pairs: 3136
Calibration genuine/impostor: {0: 3024, 1: 112}
Evaluation genuine/impostor: {0: 3024, 1: 112}

Calibration-derived threshold: 0.6722127508234095
Calibration EER (%): 32.20899470899471
Calibration error rate: 0.3223852040816326
Evaluation error rate: 0.31855867346938777


In [18]:
def build_error_predictor():
    return Pipeline(
        [
            (
                "scaler",
                StandardScaler(),
            ),
            (
                "classifier",
                LogisticRegression(
                    max_iter=2000,
                    random_state=SPLIT_SEED,
                ),
            ),
        ]
    )


calibration_error_labels = (
    uncertainty_calibration_df[
        "verification_error"
    ].to_numpy(
        dtype=np.int64
    )
)

evaluation_error_labels = (
    uncertainty_evaluation_df[
        "verification_error"
    ].to_numpy(
        dtype=np.int64
    )
)

margin_features_calibration = (
    uncertainty_calibration_df[
        [
            "margin_uncertainty"
        ]
    ].to_numpy()
)

margin_features_evaluation = (
    uncertainty_evaluation_df[
        [
            "margin_uncertainty"
        ]
    ].to_numpy()
)

reliance_features_calibration = (
    uncertainty_calibration_df[
        [
            "script_reliance"
        ]
    ].to_numpy()
)

reliance_features_evaluation = (
    uncertainty_evaluation_df[
        [
            "script_reliance"
        ]
    ].to_numpy()
)

combined_features_calibration = (
    uncertainty_calibration_df[
        [
            "margin_uncertainty",
            "script_reliance",
        ]
    ].to_numpy()
)

combined_features_evaluation = (
    uncertainty_evaluation_df[
        [
            "margin_uncertainty",
            "script_reliance",
        ]
    ].to_numpy()
)

margin_error_predictor = (
    build_error_predictor()
)

reliance_error_predictor = (
    build_error_predictor()
)

combined_error_predictor = (
    build_error_predictor()
)

margin_error_predictor.fit(
    margin_features_calibration,
    calibration_error_labels,
)

reliance_error_predictor.fit(
    reliance_features_calibration,
    calibration_error_labels,
)

combined_error_predictor.fit(
    combined_features_calibration,
    calibration_error_labels,
)

margin_error_probabilities = (
    margin_error_predictor.predict_proba(
        margin_features_evaluation
    )[
        :,
        1
    ]
)

reliance_error_probabilities = (
    reliance_error_predictor.predict_proba(
        reliance_features_evaluation
    )[
        :,
        1
    ]
)

combined_error_probabilities = (
    combined_error_predictor.predict_proba(
        combined_features_evaluation
    )[
        :,
        1
    ]
)

margin_only_auc = roc_auc_score(
    evaluation_error_labels,
    margin_error_probabilities,
)

reliance_only_auc = roc_auc_score(
    evaluation_error_labels,
    reliance_error_probabilities,
)

combined_auc = roc_auc_score(
    evaluation_error_labels,
    combined_error_probabilities,
)

combined_classifier = (
    combined_error_predictor.named_steps[
        "classifier"
    ]
)

print(
    "Evaluation margin-only error AUC:",
    margin_only_auc,
)

print(
    "Evaluation script-reliance-only error AUC:",
    reliance_only_auc,
)

print(
    "Evaluation combined error AUC:",
    combined_auc,
)

print()

print(
    "Combined gain over margin:",
    combined_auc
    - margin_only_auc,
)

print()

print(
    "Standardized margin coefficient:",
    combined_classifier.coef_[
        0,
        0,
    ],
)

print(
    "Standardized script-reliance coefficient:",
    combined_classifier.coef_[
        0,
        1,
    ],
)

Evaluation margin-only error AUC: 0.7172942713420019
Evaluation script-reliance-only error AUC: 0.5892059584151301
Evaluation combined error AUC: 0.745934985055247

Combined gain over margin: 0.02864071371324517

Standardized margin coefficient: 1.0548034101938983
Standardized script-reliance coefficient: 0.35713820675091107


In [19]:
SELECTIVE_COVERAGES = [
    1.00,
    0.90,
    0.80,
    0.70,
    0.60,
    0.50,
]


def selective_risk_summary(
    error_labels,
    risk_scores,
    coverages,
):
    error_labels = np.asarray(
        error_labels,
        dtype=np.int64,
    )

    risk_scores = np.asarray(
        risk_scores,
        dtype=np.float64,
    )

    order = np.argsort(
        risk_scores
    )

    total_pairs = len(
        error_labels
    )

    total_errors = int(
        error_labels.sum()
    )

    baseline_error_rate = (
        error_labels.mean()
    )

    rows = []

    for coverage in coverages:
        retained_count = max(
            1,
            int(
                np.floor(
                    coverage
                    * total_pairs
                )
            ),
        )

        retained_indices = order[
            :retained_count
        ]

        rejected_indices = order[
            retained_count:
        ]

        retained_error_rate = (
            error_labels[
                retained_indices
            ].mean()
        )

        rejected_errors = int(
            error_labels[
                rejected_indices
            ].sum()
        )

        if total_errors > 0:
            rejected_error_fraction = (
                rejected_errors
                / total_errors
            )
        else:
            rejected_error_fraction = np.nan

        relative_error_reduction = (
            1.0
            - retained_error_rate
            / baseline_error_rate
        )

        rows.append(
            {
                "coverage": (
                    coverage
                ),
                "retained_pairs": (
                    retained_count
                ),
                "retained_error_rate": (
                    retained_error_rate
                ),
                "relative_error_reduction": (
                    relative_error_reduction
                ),
                "rejected_error_fraction": (
                    rejected_error_fraction
                ),
            }
        )

    return pd.DataFrame(
        rows
    )


margin_selective_df = (
    selective_risk_summary(
        evaluation_error_labels,
        margin_error_probabilities,
        SELECTIVE_COVERAGES,
    )
)

combined_selective_df = (
    selective_risk_summary(
        evaluation_error_labels,
        combined_error_probabilities,
        SELECTIVE_COVERAGES,
    )
)

selective_comparison_df = pd.DataFrame(
    {
        "coverage": (
            margin_selective_df[
                "coverage"
            ]
        ),
        "margin_error_rate": (
            margin_selective_df[
                "retained_error_rate"
            ]
        ),
        "combined_error_rate": (
            combined_selective_df[
                "retained_error_rate"
            ]
        ),
        "margin_error_reduction": (
            margin_selective_df[
                "relative_error_reduction"
            ]
        ),
        "combined_error_reduction": (
            combined_selective_df[
                "relative_error_reduction"
            ]
        ),
        "margin_rejected_error_fraction": (
            margin_selective_df[
                "rejected_error_fraction"
            ]
        ),
        "combined_rejected_error_fraction": (
            combined_selective_df[
                "rejected_error_fraction"
            ]
        ),
    }
)

selective_comparison_df[
    "combined_error_rate_gain"
] = (
    selective_comparison_df[
        "margin_error_rate"
    ]
    - selective_comparison_df[
        "combined_error_rate"
    ]
)

print(
    selective_comparison_df
    .round(6)
    .to_string(
        index=False
    )
)

 coverage  margin_error_rate  combined_error_rate  margin_error_reduction  combined_error_reduction  margin_rejected_error_fraction  combined_rejected_error_fraction  combined_error_rate_gain
      1.0           0.318559             0.318559                0.000000                  0.000000                        0.000000                          0.000000                  0.000000
      0.9           0.299433             0.294826                0.060038                  0.074499                        0.154154                          0.167167                  0.004607
      0.8           0.283094             0.268740                0.111328                  0.156388                        0.289289                          0.325325                  0.014354
      0.7           0.255581             0.238269                0.197696                  0.252041                        0.438438                          0.476476                  0.017312
      0.6           0.217438            

In [22]:
def calculate_aurc(
    error_labels,
    risk_scores,
):
    error_labels = np.asarray(
        error_labels,
        dtype=np.int64,
    )

    risk_scores = np.asarray(
        risk_scores,
        dtype=np.float64,
    )

    order = np.argsort(
        risk_scores
    )

    ordered_errors = error_labels[
        order
    ]

    cumulative_errors = np.cumsum(
        ordered_errors
    )

    retained_counts = np.arange(
        1,
        len(
            ordered_errors
        )
        + 1,
    )

    coverages = (
        retained_counts
        / len(
            ordered_errors
        )
    )

    risks = (
        cumulative_errors
        / retained_counts
    )

    aurc = np.trapezoid(
        risks,
        coverages,
    )

    return float(
        aurc
    )


margin_aurc = calculate_aurc(
    evaluation_error_labels,
    margin_error_probabilities,
)

combined_aurc = calculate_aurc(
    evaluation_error_labels,
    combined_error_probabilities,
)

print(
    "Margin-only AURC:",
    margin_aurc,
)

print(
    "Combined AURC:",
    combined_aurc,
)

print(
    "AURC improvement:",
    margin_aurc
    - combined_aurc,
)

print(
    "Relative AURC improvement (%):",
    100.0
    * (
        margin_aurc
        - combined_aurc
    )
    / margin_aurc,
)

Margin-only AURC: 0.1599331403065899
Combined AURC: 0.14875280328935692
AURC improvement: 0.011180337017232977
Relative AURC improvement (%): 6.990631832652324


In [23]:
REPEATED_WRITER_SPLITS = 20


def retained_error_rate(
    error_labels,
    risk_scores,
    coverage,
):
    error_labels = np.asarray(
        error_labels,
        dtype=np.int64,
    )

    risk_scores = np.asarray(
        risk_scores,
        dtype=np.float64,
    )

    order = np.argsort(
        risk_scores
    )

    retained_count = max(
        1,
        int(
            np.floor(
                coverage
                * len(
                    error_labels
                )
            )
        ),
    )

    retained_indices = order[
        :retained_count
    ]

    return float(
        error_labels[
            retained_indices
        ].mean()
    )


def evaluate_uncertainty_writer_split(
    calibration_writers,
    evaluation_writers,
    split_index,
    fold,
):
    calibration_mask = (
        cross_script_pairs_df[
            "writer_a"
        ].isin(
            calibration_writers
        )
        & cross_script_pairs_df[
            "writer_b"
        ].isin(
            calibration_writers
        )
    )

    evaluation_mask = (
        cross_script_pairs_df[
            "writer_a"
        ].isin(
            evaluation_writers
        )
        & cross_script_pairs_df[
            "writer_b"
        ].isin(
            evaluation_writers
        )
    )

    calibration_df = (
        cross_script_pairs_df[
            calibration_mask
        ]
        .copy()
        .reset_index(drop=True)
    )

    evaluation_df = (
        cross_script_pairs_df[
            evaluation_mask
        ]
        .copy()
        .reset_index(drop=True)
    )

    calibration_eer, threshold = (
        calculate_interpolated_eer(
            calibration_df[
                "pair_label"
            ].to_numpy(
                dtype=np.int64
            ),
            calibration_df[
                "original_score"
            ].to_numpy(),
        )
    )

    for dataframe in [
        calibration_df,
        evaluation_df,
    ]:
        dataframe[
            "verification_error"
        ] = (
            (
                dataframe[
                    "original_score"
                ]
                >= threshold
            ).astype(
                np.int64
            )
            != dataframe[
                "pair_label"
            ].to_numpy(
                dtype=np.int64
            )
        ).astype(
            np.int64
        )

        dataframe[
            "margin_uncertainty"
        ] = (
            -np.abs(
                dataframe[
                    "original_score"
                ]
                - threshold
            )
        )

    calibration_errors = (
        calibration_df[
            "verification_error"
        ].to_numpy(
            dtype=np.int64
        )
    )

    evaluation_errors = (
        evaluation_df[
            "verification_error"
        ].to_numpy(
            dtype=np.int64
        )
    )

    margin_model = (
        build_error_predictor()
    )

    reliance_model = (
        build_error_predictor()
    )

    combined_model = (
        build_error_predictor()
    )

    margin_model.fit(
        calibration_df[
            [
                "margin_uncertainty"
            ]
        ].to_numpy(),
        calibration_errors,
    )

    reliance_model.fit(
        calibration_df[
            [
                "script_reliance"
            ]
        ].to_numpy(),
        calibration_errors,
    )

    combined_model.fit(
        calibration_df[
            [
                "margin_uncertainty",
                "script_reliance",
            ]
        ].to_numpy(),
        calibration_errors,
    )

    margin_risk = (
        margin_model.predict_proba(
            evaluation_df[
                [
                    "margin_uncertainty"
                ]
            ].to_numpy()
        )[
            :,
            1
        ]
    )

    reliance_risk = (
        reliance_model.predict_proba(
            evaluation_df[
                [
                    "script_reliance"
                ]
            ].to_numpy()
        )[
            :,
            1
        ]
    )

    combined_risk = (
        combined_model.predict_proba(
            evaluation_df[
                [
                    "margin_uncertainty",
                    "script_reliance",
                ]
            ].to_numpy()
        )[
            :,
            1
        ]
    )

    margin_auc = roc_auc_score(
        evaluation_errors,
        margin_risk,
    )

    reliance_auc = roc_auc_score(
        evaluation_errors,
        reliance_risk,
    )

    combined_auc = roc_auc_score(
        evaluation_errors,
        combined_risk,
    )

    margin_aurc = calculate_aurc(
        evaluation_errors,
        margin_risk,
    )

    combined_aurc = calculate_aurc(
        evaluation_errors,
        combined_risk,
    )

    margin_error_80 = retained_error_rate(
        evaluation_errors,
        margin_risk,
        0.80,
    )

    combined_error_80 = retained_error_rate(
        evaluation_errors,
        combined_risk,
        0.80,
    )

    margin_error_50 = retained_error_rate(
        evaluation_errors,
        margin_risk,
        0.50,
    )

    combined_error_50 = retained_error_rate(
        evaluation_errors,
        combined_risk,
        0.50,
    )

    return {
        "split_index": (
            split_index
        ),
        "fold": (
            fold
        ),
        "calibration_threshold": (
            threshold
        ),
        "calibration_eer": (
            calibration_eer
        ),
        "evaluation_error_rate": (
            evaluation_errors.mean()
        ),
        "margin_auc": (
            margin_auc
        ),
        "reliance_auc": (
            reliance_auc
        ),
        "combined_auc": (
            combined_auc
        ),
        "combined_auc_gain": (
            combined_auc
            - margin_auc
        ),
        "margin_aurc": (
            margin_aurc
        ),
        "combined_aurc": (
            combined_aurc
        ),
        "aurc_improvement": (
            margin_aurc
            - combined_aurc
        ),
        "margin_error_80": (
            margin_error_80
        ),
        "combined_error_80": (
            combined_error_80
        ),
        "error_gain_80": (
            margin_error_80
            - combined_error_80
        ),
        "margin_error_50": (
            margin_error_50
        ),
        "combined_error_50": (
            combined_error_50
        ),
        "error_gain_50": (
            margin_error_50
            - combined_error_50
        ),
    }


repeated_uncertainty_rows = []

for split_index in range(
    REPEATED_WRITER_SPLITS
):
    rng = np.random.default_rng(
        1000
        + SPLIT_SEED
        + split_index
    )

    shuffled_writers = rng.permutation(
        validation_writers
    )

    writer_half_a = shuffled_writers[
        :28
    ]

    writer_half_b = shuffled_writers[
        28:
    ]

    repeated_uncertainty_rows.append(
        evaluate_uncertainty_writer_split(
            writer_half_a,
            writer_half_b,
            split_index,
            "A_to_B",
        )
    )

    repeated_uncertainty_rows.append(
        evaluate_uncertainty_writer_split(
            writer_half_b,
            writer_half_a,
            split_index,
            "B_to_A",
        )
    )


repeated_uncertainty_df = pd.DataFrame(
    repeated_uncertainty_rows
)

print(
    "Writer-disjoint evaluations:",
    len(
        repeated_uncertainty_df
    ),
)

print(
    repeated_uncertainty_df[
        [
            "split_index",
            "fold",
            "margin_auc",
            "reliance_auc",
            "combined_auc",
            "combined_auc_gain",
            "aurc_improvement",
        ]
    ]
    .round(6)
    .to_string(
        index=False
    )
)

Writer-disjoint evaluations: 40
 split_index   fold  margin_auc  reliance_auc  combined_auc  combined_auc_gain  aurc_improvement
           0 A_to_B    0.718683      0.575003      0.739943           0.021260          0.007819
           0 B_to_A    0.727899      0.608254      0.753606           0.025707          0.010277
           1 A_to_B    0.710595      0.558598      0.736798           0.026203          0.009664
           1 B_to_A    0.733363      0.608302      0.750689           0.017326          0.006022
           2 A_to_B    0.765529      0.602753      0.778414           0.012885          0.004260
           2 B_to_A    0.677169      0.576910      0.714562           0.037394          0.016546
           3 A_to_B    0.748367      0.625753      0.771066           0.022699          0.008345
           3 B_to_A    0.708447      0.567464      0.732512           0.024066          0.010175
           4 A_to_B    0.740495      0.592401      0.765156           0.024662          0.00906

In [24]:
print(
    "Mean margin-only error AUC:",
    repeated_uncertainty_df[
        "margin_auc"
    ].mean(),
)

print(
    "Mean script-reliance-only error AUC:",
    repeated_uncertainty_df[
        "reliance_auc"
    ].mean(),
)

print(
    "Mean combined error AUC:",
    repeated_uncertainty_df[
        "combined_auc"
    ].mean(),
)

print()

print(
    "Mean combined AUC gain:",
    repeated_uncertainty_df[
        "combined_auc_gain"
    ].mean(),
)

print(
    "Median combined AUC gain:",
    repeated_uncertainty_df[
        "combined_auc_gain"
    ].median(),
)

print(
    "Combined AUC better:",
    int(
        (
            repeated_uncertainty_df[
                "combined_auc_gain"
            ] > 0
        ).sum()
    ),
    "/",
    len(
        repeated_uncertainty_df
    ),
)

print()

print(
    "Mean margin-only AURC:",
    repeated_uncertainty_df[
        "margin_aurc"
    ].mean(),
)

print(
    "Mean combined AURC:",
    repeated_uncertainty_df[
        "combined_aurc"
    ].mean(),
)

print(
    "Mean AURC improvement:",
    repeated_uncertainty_df[
        "aurc_improvement"
    ].mean(),
)

print(
    "Combined AURC better:",
    int(
        (
            repeated_uncertainty_df[
                "aurc_improvement"
            ] > 0
        ).sum()
    ),
    "/",
    len(
        repeated_uncertainty_df
    ),
)

print()

print(
    "Mean error-rate gain at 80% coverage:",
    repeated_uncertainty_df[
        "error_gain_80"
    ].mean(),
)

print(
    "Combined better at 80% coverage:",
    int(
        (
            repeated_uncertainty_df[
                "error_gain_80"
            ] > 0
        ).sum()
    ),
    "/",
    len(
        repeated_uncertainty_df
    ),
)

print()

print(
    "Mean error-rate gain at 50% coverage:",
    repeated_uncertainty_df[
        "error_gain_50"
    ].mean(),
)

print(
    "Combined better at 50% coverage:",
    int(
        (
            repeated_uncertainty_df[
                "error_gain_50"
            ] > 0
        ).sum()
    ),
    "/",
    len(
        repeated_uncertainty_df
    ),
)

Mean margin-only error AUC: 0.72640788727036
Mean script-reliance-only error AUC: 0.5929739601050439
Mean combined error AUC: 0.7493880068716661

Mean combined AUC gain: 0.022980119601306116
Median combined AUC gain: 0.022638206719368392
Combined AUC better: 40 / 40

Mean margin-only AURC: 0.15970908480278187
Mean combined AURC: 0.15075828122930052
Mean AURC improvement: 0.008950803573481391
Combined AURC better: 40 / 40

Mean error-rate gain at 80% coverage: 0.008193779904306222
Combined better at 80% coverage: 40 / 40

Mean error-rate gain at 50% coverage: 0.019818239795918367
Combined better at 50% coverage: 40 / 40


In [25]:
seed2026_checkpoint_path = next(
    path
    for path in batch_candidates
    if path.name
    == "seed2026_batch_alt_lambda0p5_best.pt"
)

seed2026_checkpoint = torch.load(
    seed2026_checkpoint_path,
    map_location="cpu",
    weights_only=False,
)

seed2026_model = (
    ScriptAdversarialWriterModel()
    .to(
        TRAIN_DEVICE
    )
)

seed2026_model.load_state_dict(
    seed2026_checkpoint[
        "model_state_dict"
    ],
    strict=True,
)

seed2026_model.eval()

development_embeddings_2026, development_script_labels_2026, development_filenames_2026 = (
    extract_embeddings_and_labels(
        seed2026_model,
        development_loader,
    )
)

validation_embeddings_2026, validation_script_labels_2026, validation_filenames_2026 = (
    extract_embeddings_and_labels(
        seed2026_model,
        validation_loader,
    )
)

intervened_development_2026 = (
    development_embeddings_2026.copy()
)

intervened_validation_2026 = (
    validation_embeddings_2026.copy()
)

script_basis_2026 = []

for dimension in range(
    1,
    SELECTED_SCRIPT_SUBSPACE_DIMENSION + 1,
):
    _, raw_direction = (
        fit_effective_script_direction(
            intervened_development_2026,
            development_script_labels_2026,
        )
    )

    residual_direction = (
        raw_direction.copy()
    )

    for basis_direction in (
        script_basis_2026
    ):
        residual_direction = (
            residual_direction
            - np.dot(
                residual_direction,
                basis_direction,
            )
            * basis_direction
        )

    residual_direction = (
        residual_direction
        / np.linalg.norm(
            residual_direction
        )
    )

    script_basis_2026.append(
        residual_direction
    )

    intervened_development_2026, _ = (
        remove_script_direction(
            intervened_development_2026,
            residual_direction,
        )
    )

    intervened_validation_2026, _ = (
        remove_script_direction(
            intervened_validation_2026,
            residual_direction,
        )
    )


residual_probe_2026 = Pipeline(
    [
        (
            "scaler",
            StandardScaler(),
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                random_state=PROBE_SEED,
            ),
        ),
    ]
)

residual_probe_2026.fit(
    intervened_development_2026,
    development_script_labels_2026,
)

residual_probabilities_2026 = (
    residual_probe_2026.predict_proba(
        intervened_validation_2026
    )[
        :,
        1
    ]
)

residual_script_auc_2026 = (
    roc_auc_score(
        validation_script_labels_2026,
        residual_probabilities_2026,
    )
)

original_map_2026 = dict(
    zip(
        validation_filenames_2026,
        validation_embeddings_2026,
    )
)

intervened_map_2026 = dict(
    zip(
        validation_filenames_2026,
        intervened_validation_2026,
    )
)

original_a_2026 = np.stack(
    [
        original_map_2026[
            filename
        ]
        for filename in (
            validation_pairs_df[
                "filename_a"
            ]
        )
    ]
)

original_b_2026 = np.stack(
    [
        original_map_2026[
            filename
        ]
        for filename in (
            validation_pairs_df[
                "filename_b"
            ]
        )
    ]
)

intervened_a_2026 = np.stack(
    [
        intervened_map_2026[
            filename
        ]
        for filename in (
            validation_pairs_df[
                "filename_a"
            ]
        )
    ]
)

intervened_b_2026 = np.stack(
    [
        intervened_map_2026[
            filename
        ]
        for filename in (
            validation_pairs_df[
                "filename_b"
            ]
        )
    ]
)

original_scores_2026 = np.sum(
    original_a_2026
    * original_b_2026,
    axis=1,
)

intervened_scores_2026 = np.sum(
    intervened_a_2026
    * intervened_b_2026,
    axis=1,
)

pair_effect_2026_df = (
    validation_pairs_df
    .copy()
    .reset_index(drop=True)
)

pair_effect_2026_df[
    "original_score"
] = original_scores_2026

pair_effect_2026_df[
    "intervened_score"
] = intervened_scores_2026

pair_effect_2026_df[
    "score_shift"
] = (
    intervened_scores_2026
    - original_scores_2026
)

pair_effect_2026_df[
    "script_reliance"
] = np.abs(
    pair_effect_2026_df[
        "score_shift"
    ]
)

pair_effect_2026_df[
    "page_a"
] = pair_effect_2026_df[
    "filename_a"
].map(
    filename_page_id
)

pair_effect_2026_df[
    "page_b"
] = pair_effect_2026_df[
    "filename_b"
].map(
    filename_page_id
)

pair_effect_2026_df[
    "condition"
] = [
    CONDITION_BY_PAGE_PAIR[
        (
            int(
                page_a
            ),
            int(
                page_b
            ),
        )
    ]
    for page_a, page_b in zip(
        pair_effect_2026_df[
            "page_a"
        ],
        pair_effect_2026_df[
            "page_b"
        ],
    )
]

pair_effect_2026_df[
    "family"
] = np.where(
    pair_effect_2026_df[
        "condition"
    ].str.startswith(
        "cross_"
    ),
    "cross_script",
    "within_script",
)

pair_effect_2026_df[
    "writer_a"
] = pair_effect_2026_df[
    "filename_a"
].map(
    validation_writer_map
)

pair_effect_2026_df[
    "writer_b"
] = pair_effect_2026_df[
    "filename_b"
].map(
    validation_writer_map
)

original_auc_2026 = roc_auc_score(
    pair_effect_2026_df[
        "pair_label"
    ],
    pair_effect_2026_df[
        "original_score"
    ],
)

intervened_auc_2026 = roc_auc_score(
    pair_effect_2026_df[
        "pair_label"
    ],
    pair_effect_2026_df[
        "intervened_score"
    ],
)

print(
    "Seed-2026 checkpoint AUC:",
    seed2026_checkpoint[
        "validation_auc"
    ],
)

print(
    "Seed-2026 reproduced AUC:",
    original_auc_2026,
)

print(
    "Residual script AUC after fixed 128-D intervention:",
    residual_script_auc_2026,
)

print(
    "Intervened writer AUC:",
    intervened_auc_2026,
)

print(
    "Mean script reliance:",
    pair_effect_2026_df[
        "script_reliance"
    ].mean(),
)

Seed-2026 checkpoint AUC: 0.7700305832560297
Seed-2026 reproduced AUC: 0.7700305832560297
Residual script AUC after fixed 128-D intervention: 0.7010522959183674
Intervened writer AUC: 0.7706360479797979
Mean script reliance: 0.010207820265921593


In [26]:
cross_script_pairs_2026_df = (
    pair_effect_2026_df[
        pair_effect_2026_df[
            "family"
        ] == "cross_script"
    ]
    .copy()
    .reset_index(drop=True)
)


def evaluate_repeated_uncertainty_for_pairs(
    cross_pairs_df,
):
    rows = []

    for split_index in range(
        REPEATED_WRITER_SPLITS
    ):
        rng = np.random.default_rng(
            1000
            + SPLIT_SEED
            + split_index
        )

        shuffled_writers = (
            rng.permutation(
                validation_writers
            )
        )

        writer_half_a = (
            shuffled_writers[
                :28
            ]
        )

        writer_half_b = (
            shuffled_writers[
                28:
            ]
        )

        for fold, calibration_writers, evaluation_writers_current in [
            (
                "A_to_B",
                writer_half_a,
                writer_half_b,
            ),
            (
                "B_to_A",
                writer_half_b,
                writer_half_a,
            ),
        ]:
            calibration_mask = (
                cross_pairs_df[
                    "writer_a"
                ].isin(
                    calibration_writers
                )
                & cross_pairs_df[
                    "writer_b"
                ].isin(
                    calibration_writers
                )
            )

            evaluation_mask = (
                cross_pairs_df[
                    "writer_a"
                ].isin(
                    evaluation_writers_current
                )
                & cross_pairs_df[
                    "writer_b"
                ].isin(
                    evaluation_writers_current
                )
            )

            calibration_df = (
                cross_pairs_df[
                    calibration_mask
                ]
                .copy()
                .reset_index(drop=True)
            )

            evaluation_df = (
                cross_pairs_df[
                    evaluation_mask
                ]
                .copy()
                .reset_index(drop=True)
            )

            _, threshold = (
                calculate_interpolated_eer(
                    calibration_df[
                        "pair_label"
                    ].to_numpy(
                        dtype=np.int64
                    ),
                    calibration_df[
                        "original_score"
                    ].to_numpy(),
                )
            )

            for dataframe in [
                calibration_df,
                evaluation_df,
            ]:
                dataframe[
                    "verification_error"
                ] = (
                    (
                        dataframe[
                            "original_score"
                        ]
                        >= threshold
                    ).astype(
                        np.int64
                    )
                    != dataframe[
                        "pair_label"
                    ].to_numpy(
                        dtype=np.int64
                    )
                ).astype(
                    np.int64
                )

                dataframe[
                    "margin_uncertainty"
                ] = (
                    -np.abs(
                        dataframe[
                            "original_score"
                        ]
                        - threshold
                    )
                )

            calibration_errors = (
                calibration_df[
                    "verification_error"
                ].to_numpy(
                    dtype=np.int64
                )
            )

            evaluation_errors = (
                evaluation_df[
                    "verification_error"
                ].to_numpy(
                    dtype=np.int64
                )
            )

            margin_model = (
                build_error_predictor()
            )

            reliance_model = (
                build_error_predictor()
            )

            combined_model = (
                build_error_predictor()
            )

            margin_model.fit(
                calibration_df[
                    [
                        "margin_uncertainty"
                    ]
                ].to_numpy(),
                calibration_errors,
            )

            reliance_model.fit(
                calibration_df[
                    [
                        "script_reliance"
                    ]
                ].to_numpy(),
                calibration_errors,
            )

            combined_model.fit(
                calibration_df[
                    [
                        "margin_uncertainty",
                        "script_reliance",
                    ]
                ].to_numpy(),
                calibration_errors,
            )

            margin_risk = (
                margin_model.predict_proba(
                    evaluation_df[
                        [
                            "margin_uncertainty"
                        ]
                    ].to_numpy()
                )[
                    :,
                    1
                ]
            )

            reliance_risk = (
                reliance_model.predict_proba(
                    evaluation_df[
                        [
                            "script_reliance"
                        ]
                    ].to_numpy()
                )[
                    :,
                    1
                ]
            )

            combined_risk = (
                combined_model.predict_proba(
                    evaluation_df[
                        [
                            "margin_uncertainty",
                            "script_reliance",
                        ]
                    ].to_numpy()
                )[
                    :,
                    1
                ]
            )

            margin_auc = roc_auc_score(
                evaluation_errors,
                margin_risk,
            )

            reliance_auc = roc_auc_score(
                evaluation_errors,
                reliance_risk,
            )

            combined_auc = roc_auc_score(
                evaluation_errors,
                combined_risk,
            )

            margin_aurc = calculate_aurc(
                evaluation_errors,
                margin_risk,
            )

            combined_aurc = calculate_aurc(
                evaluation_errors,
                combined_risk,
            )

            margin_error_80 = (
                retained_error_rate(
                    evaluation_errors,
                    margin_risk,
                    0.80,
                )
            )

            combined_error_80 = (
                retained_error_rate(
                    evaluation_errors,
                    combined_risk,
                    0.80,
                )
            )

            margin_error_50 = (
                retained_error_rate(
                    evaluation_errors,
                    margin_risk,
                    0.50,
                )
            )

            combined_error_50 = (
                retained_error_rate(
                    evaluation_errors,
                    combined_risk,
                    0.50,
                )
            )

            rows.append(
                {
                    "split_index": (
                        split_index
                    ),
                    "fold": (
                        fold
                    ),
                    "margin_auc": (
                        margin_auc
                    ),
                    "reliance_auc": (
                        reliance_auc
                    ),
                    "combined_auc": (
                        combined_auc
                    ),
                    "combined_auc_gain": (
                        combined_auc
                        - margin_auc
                    ),
                    "margin_aurc": (
                        margin_aurc
                    ),
                    "combined_aurc": (
                        combined_aurc
                    ),
                    "aurc_improvement": (
                        margin_aurc
                        - combined_aurc
                    ),
                    "error_gain_80": (
                        margin_error_80
                        - combined_error_80
                    ),
                    "error_gain_50": (
                        margin_error_50
                        - combined_error_50
                    ),
                }
            )

    return pd.DataFrame(
        rows
    )


repeated_uncertainty_2026_df = (
    evaluate_repeated_uncertainty_for_pairs(
        cross_script_pairs_2026_df
    )
)

print(
    "Writer-disjoint evaluations:",
    len(
        repeated_uncertainty_2026_df
    ),
)

print()

print(
    "Mean margin-only error AUC:",
    repeated_uncertainty_2026_df[
        "margin_auc"
    ].mean(),
)

print(
    "Mean script-reliance-only error AUC:",
    repeated_uncertainty_2026_df[
        "reliance_auc"
    ].mean(),
)

print(
    "Mean combined error AUC:",
    repeated_uncertainty_2026_df[
        "combined_auc"
    ].mean(),
)

print(
    "Mean combined AUC gain:",
    repeated_uncertainty_2026_df[
        "combined_auc_gain"
    ].mean(),
)

print(
    "Combined AUC better:",
    int(
        (
            repeated_uncertainty_2026_df[
                "combined_auc_gain"
            ] > 0
        ).sum()
    ),
    "/",
    len(
        repeated_uncertainty_2026_df
    ),
)

print()

print(
    "Mean AURC improvement:",
    repeated_uncertainty_2026_df[
        "aurc_improvement"
    ].mean(),
)

print(
    "Combined AURC better:",
    int(
        (
            repeated_uncertainty_2026_df[
                "aurc_improvement"
            ] > 0
        ).sum()
    ),
    "/",
    len(
        repeated_uncertainty_2026_df
    ),
)

print()

print(
    "Mean error-rate gain at 80% coverage:",
    repeated_uncertainty_2026_df[
        "error_gain_80"
    ].mean(),
)

print(
    "Combined better at 80% coverage:",
    int(
        (
            repeated_uncertainty_2026_df[
                "error_gain_80"
            ] > 0
        ).sum()
    ),
    "/",
    len(
        repeated_uncertainty_2026_df
    ),
)

print()

print(
    "Mean error-rate gain at 50% coverage:",
    repeated_uncertainty_2026_df[
        "error_gain_50"
    ].mean(),
)

print(
    "Combined better at 50% coverage:",
    int(
        (
            repeated_uncertainty_2026_df[
                "error_gain_50"
            ] > 0
        ).sum()
    ),
    "/",
    len(
        repeated_uncertainty_2026_df
    ),
)

Writer-disjoint evaluations: 40

Mean margin-only error AUC: 0.7058164529938346
Mean script-reliance-only error AUC: 0.6514476006907628
Mean combined error AUC: 0.7234088916630583
Mean combined AUC gain: 0.017592438669223772
Combined AUC better: 39 / 40

Mean AURC improvement: 0.007645061116021985
Combined AURC better: 39 / 40

Mean error-rate gain at 80% coverage: 0.006130382775119613
Combined better at 80% coverage: 35 / 40

Mean error-rate gain at 50% coverage: 0.014636479591836737
Combined better at 50% coverage: 39 / 40


In [27]:
def build_script_reliance_pair_effect(
    checkpoint_path,
):
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )

    model = (
        ScriptAdversarialWriterModel()
        .to(
            TRAIN_DEVICE
        )
    )

    model.load_state_dict(
        checkpoint[
            "model_state_dict"
        ],
        strict=True,
    )

    model.eval()

    development_embeddings_current, development_labels_current, development_filenames_current = (
        extract_embeddings_and_labels(
            model,
            development_loader,
        )
    )

    validation_embeddings_current, validation_labels_current, validation_filenames_current = (
        extract_embeddings_and_labels(
            model,
            validation_loader,
        )
    )

    intervened_development = (
        development_embeddings_current.copy()
    )

    intervened_validation = (
        validation_embeddings_current.copy()
    )

    basis = []

    for dimension in range(
        1,
        SELECTED_SCRIPT_SUBSPACE_DIMENSION + 1,
    ):
        _, raw_direction = (
            fit_effective_script_direction(
                intervened_development,
                development_labels_current,
            )
        )

        residual_direction = (
            raw_direction.copy()
        )

        for basis_direction in basis:
            residual_direction = (
                residual_direction
                - np.dot(
                    residual_direction,
                    basis_direction,
                )
                * basis_direction
            )

        residual_direction = (
            residual_direction
            / np.linalg.norm(
                residual_direction
            )
        )

        basis.append(
            residual_direction
        )

        intervened_development, _ = (
            remove_script_direction(
                intervened_development,
                residual_direction,
            )
        )

        intervened_validation, _ = (
            remove_script_direction(
                intervened_validation,
                residual_direction,
            )
        )

    residual_probe = Pipeline(
        [
            (
                "scaler",
                StandardScaler(),
            ),
            (
                "classifier",
                LogisticRegression(
                    max_iter=2000,
                    random_state=PROBE_SEED,
                ),
            ),
        ]
    )

    residual_probe.fit(
        intervened_development,
        development_labels_current,
    )

    residual_probabilities = (
        residual_probe.predict_proba(
            intervened_validation
        )[
            :,
            1
        ]
    )

    residual_script_auc = (
        roc_auc_score(
            validation_labels_current,
            residual_probabilities,
        )
    )

    original_map = dict(
        zip(
            validation_filenames_current,
            validation_embeddings_current,
        )
    )

    intervened_map = dict(
        zip(
            validation_filenames_current,
            intervened_validation,
        )
    )

    original_a = np.stack(
        [
            original_map[
                filename
            ]
            for filename in (
                validation_pairs_df[
                    "filename_a"
                ]
            )
        ]
    )

    original_b = np.stack(
        [
            original_map[
                filename
            ]
            for filename in (
                validation_pairs_df[
                    "filename_b"
                ]
            )
        ]
    )

    intervened_a = np.stack(
        [
            intervened_map[
                filename
            ]
            for filename in (
                validation_pairs_df[
                    "filename_a"
                ]
            )
        ]
    )

    intervened_b = np.stack(
        [
            intervened_map[
                filename
            ]
            for filename in (
                validation_pairs_df[
                    "filename_b"
                ]
            )
        ]
    )

    original_scores_current = np.sum(
        original_a
        * original_b,
        axis=1,
    )

    intervened_scores_current = np.sum(
        intervened_a
        * intervened_b,
        axis=1,
    )

    pair_df = (
        validation_pairs_df
        .copy()
        .reset_index(drop=True)
    )

    pair_df[
        "original_score"
    ] = original_scores_current

    pair_df[
        "intervened_score"
    ] = intervened_scores_current

    pair_df[
        "score_shift"
    ] = (
        intervened_scores_current
        - original_scores_current
    )

    pair_df[
        "script_reliance"
    ] = np.abs(
        pair_df[
            "score_shift"
        ]
    )

    pair_df[
        "page_a"
    ] = pair_df[
        "filename_a"
    ].map(
        filename_page_id
    )

    pair_df[
        "page_b"
    ] = pair_df[
        "filename_b"
    ].map(
        filename_page_id
    )

    pair_df[
        "condition"
    ] = [
        CONDITION_BY_PAGE_PAIR[
            (
                int(
                    page_a
                ),
                int(
                    page_b
                ),
            )
        ]
        for page_a, page_b in zip(
            pair_df[
                "page_a"
            ],
            pair_df[
                "page_b"
            ],
        )
    ]

    pair_df[
        "family"
    ] = np.where(
        pair_df[
            "condition"
        ].str.startswith(
            "cross_"
        ),
        "cross_script",
        "within_script",
    )

    pair_df[
        "writer_a"
    ] = pair_df[
        "filename_a"
    ].map(
        validation_writer_map
    )

    pair_df[
        "writer_b"
    ] = pair_df[
        "filename_b"
    ].map(
        validation_writer_map
    )

    original_auc_current = roc_auc_score(
        pair_df[
            "pair_label"
        ],
        pair_df[
            "original_score"
        ],
    )

    intervened_auc_current = roc_auc_score(
        pair_df[
            "pair_label"
        ],
        pair_df[
            "intervened_score"
        ],
    )

    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return (
        pair_df,
        {
            "residual_script_auc": float(
                residual_script_auc
            ),
            "original_writer_auc": float(
                original_auc_current
            ),
            "intervened_writer_auc": float(
                intervened_auc_current
            ),
            "mean_script_reliance": float(
                pair_df[
                    "script_reliance"
                ].mean()
            ),
        },
    )


print(
    "Generic Script Reliance evaluator ready."
)

Generic Script Reliance evaluator ready.


In [28]:
seed123_control_checkpoint_path = next(
    path
    for path in checkpoint_candidates
    if path.name
    == "seed123_lambda0_control_best.pt"
)

seed123_control_pair_df, seed123_control_intervention = (
    build_script_reliance_pair_effect(
        seed123_control_checkpoint_path
    )
)

seed123_control_cross_df = (
    seed123_control_pair_df[
        seed123_control_pair_df[
            "family"
        ] == "cross_script"
    ]
    .copy()
    .reset_index(drop=True)
)

seed123_control_uncertainty_df = (
    evaluate_repeated_uncertainty_for_pairs(
        seed123_control_cross_df
    )
)

print(
    "Control original writer AUC:",
    seed123_control_intervention[
        "original_writer_auc"
    ],
)

print(
    "Control intervened writer AUC:",
    seed123_control_intervention[
        "intervened_writer_auc"
    ],
)

print(
    "Residual script AUC:",
    seed123_control_intervention[
        "residual_script_auc"
    ],
)

print(
    "Mean script reliance:",
    seed123_control_intervention[
        "mean_script_reliance"
    ],
)

print()

print(
    "Mean margin-only error AUC:",
    seed123_control_uncertainty_df[
        "margin_auc"
    ].mean(),
)

print(
    "Mean script-reliance-only error AUC:",
    seed123_control_uncertainty_df[
        "reliance_auc"
    ].mean(),
)

print(
    "Mean combined error AUC:",
    seed123_control_uncertainty_df[
        "combined_auc"
    ].mean(),
)

print(
    "Mean combined AUC gain:",
    seed123_control_uncertainty_df[
        "combined_auc_gain"
    ].mean(),
)

print(
    "Combined AUC better:",
    int(
        (
            seed123_control_uncertainty_df[
                "combined_auc_gain"
            ] > 0
        ).sum()
    ),
    "/",
    len(
        seed123_control_uncertainty_df
    ),
)

print()

print(
    "Mean AURC improvement:",
    seed123_control_uncertainty_df[
        "aurc_improvement"
    ].mean(),
)

print(
    "Combined AURC better:",
    int(
        (
            seed123_control_uncertainty_df[
                "aurc_improvement"
            ] > 0
        ).sum()
    ),
    "/",
    len(
        seed123_control_uncertainty_df
    ),
)

Control original writer AUC: 0.7387890383426098
Control intervened writer AUC: 0.768275387806638
Residual script AUC: 0.8911033163265305
Mean script reliance: 0.06332696854855677

Mean margin-only error AUC: 0.6632703053875306
Mean script-reliance-only error AUC: 0.6172194767667678
Mean combined error AUC: 0.6835319065357662
Mean combined AUC gain: 0.020261601148235595
Combined AUC better: 35 / 40

Mean AURC improvement: 0.019516576280170284
Combined AURC better: 37 / 40


In [29]:
seed2026_control_checkpoint_path = next(
    path
    for path in checkpoint_candidates
    if path.name
    == "seed2026_lambda0_control_best.pt"
)

seed2026_control_pair_df, seed2026_control_intervention = (
    build_script_reliance_pair_effect(
        seed2026_control_checkpoint_path
    )
)

seed2026_control_cross_df = (
    seed2026_control_pair_df[
        seed2026_control_pair_df[
            "family"
        ] == "cross_script"
    ]
    .copy()
    .reset_index(drop=True)
)

seed2026_control_uncertainty_df = (
    evaluate_repeated_uncertainty_for_pairs(
        seed2026_control_cross_df
    )
)

print(
    "Control original writer AUC:",
    seed2026_control_intervention[
        "original_writer_auc"
    ],
)

print(
    "Control intervened writer AUC:",
    seed2026_control_intervention[
        "intervened_writer_auc"
    ],
)

print(
    "Residual script AUC:",
    seed2026_control_intervention[
        "residual_script_auc"
    ],
)

print(
    "Mean script reliance:",
    seed2026_control_intervention[
        "mean_script_reliance"
    ],
)

print()

print(
    "Mean margin-only error AUC:",
    seed2026_control_uncertainty_df[
        "margin_auc"
    ].mean(),
)

print(
    "Mean script-reliance-only error AUC:",
    seed2026_control_uncertainty_df[
        "reliance_auc"
    ].mean(),
)

print(
    "Mean combined error AUC:",
    seed2026_control_uncertainty_df[
        "combined_auc"
    ].mean(),
)

print(
    "Mean combined AUC gain:",
    seed2026_control_uncertainty_df[
        "combined_auc_gain"
    ].mean(),
)

print(
    "Combined AUC better:",
    int(
        (
            seed2026_control_uncertainty_df[
                "combined_auc_gain"
            ] > 0
        ).sum()
    ),
    "/",
    len(
        seed2026_control_uncertainty_df
    ),
)

print()

print(
    "Mean AURC improvement:",
    seed2026_control_uncertainty_df[
        "aurc_improvement"
    ].mean(),
)

print(
    "Combined AURC better:",
    int(
        (
            seed2026_control_uncertainty_df[
                "aurc_improvement"
            ] > 0
        ).sum()
    ),
    "/",
    len(
        seed2026_control_uncertainty_df
    ),
)

Control original writer AUC: 0.7382791563595135
Control intervened writer AUC: 0.762909870387549
Residual script AUC: 0.9269770408163266
Mean script reliance: 0.03647650955010961

Mean margin-only error AUC: 0.6764488163212936
Mean script-reliance-only error AUC: 0.4766655325888668
Mean combined error AUC: 0.6729641764892484
Mean combined AUC gain: -0.003484639832045161
Combined AUC better: 10 / 40

Mean AURC improvement: -0.0015686736108587427
Combined AURC better: 14 / 40


In [30]:
model_reliance_summary_df = pd.DataFrame(
    [
        {
            "training_seed": 123,
            "model": "lambda0_control",
            "original_writer_auc": (
                seed123_control_intervention[
                    "original_writer_auc"
                ]
            ),
            "intervened_writer_auc": (
                seed123_control_intervention[
                    "intervened_writer_auc"
                ]
            ),
            "residual_script_auc": (
                seed123_control_intervention[
                    "residual_script_auc"
                ]
            ),
            "mean_script_reliance": (
                seed123_control_intervention[
                    "mean_script_reliance"
                ]
            ),
            "reliance_error_auc": (
                seed123_control_uncertainty_df[
                    "reliance_auc"
                ].mean()
            ),
            "combined_auc_gain": (
                seed123_control_uncertainty_df[
                    "combined_auc_gain"
                ].mean()
            ),
            "combined_auc_better": int(
                (
                    seed123_control_uncertainty_df[
                        "combined_auc_gain"
                    ] > 0
                ).sum()
            ),
            "aurc_better": int(
                (
                    seed123_control_uncertainty_df[
                        "aurc_improvement"
                    ] > 0
                ).sum()
            ),
        },
        {
            "training_seed": 123,
            "model": "batch_alt_lambda0p5",
            "original_writer_auc": (
                original_auc
            ),
            "intervened_writer_auc": (
                intervened_auc
            ),
            "residual_script_auc": (
                final_validation_script_auc
            ),
            "mean_script_reliance": (
                pair_effect_df[
                    "script_reliance"
                ].mean()
            ),
            "reliance_error_auc": (
                repeated_uncertainty_df[
                    "reliance_auc"
                ].mean()
            ),
            "combined_auc_gain": (
                repeated_uncertainty_df[
                    "combined_auc_gain"
                ].mean()
            ),
            "combined_auc_better": int(
                (
                    repeated_uncertainty_df[
                        "combined_auc_gain"
                    ] > 0
                ).sum()
            ),
            "aurc_better": int(
                (
                    repeated_uncertainty_df[
                        "aurc_improvement"
                    ] > 0
                ).sum()
            ),
        },
        {
            "training_seed": 2026,
            "model": "lambda0_control",
            "original_writer_auc": (
                seed2026_control_intervention[
                    "original_writer_auc"
                ]
            ),
            "intervened_writer_auc": (
                seed2026_control_intervention[
                    "intervened_writer_auc"
                ]
            ),
            "residual_script_auc": (
                seed2026_control_intervention[
                    "residual_script_auc"
                ]
            ),
            "mean_script_reliance": (
                seed2026_control_intervention[
                    "mean_script_reliance"
                ]
            ),
            "reliance_error_auc": (
                seed2026_control_uncertainty_df[
                    "reliance_auc"
                ].mean()
            ),
            "combined_auc_gain": (
                seed2026_control_uncertainty_df[
                    "combined_auc_gain"
                ].mean()
            ),
            "combined_auc_better": int(
                (
                    seed2026_control_uncertainty_df[
                        "combined_auc_gain"
                    ] > 0
                ).sum()
            ),
            "aurc_better": int(
                (
                    seed2026_control_uncertainty_df[
                        "aurc_improvement"
                    ] > 0
                ).sum()
            ),
        },
        {
            "training_seed": 2026,
            "model": "batch_alt_lambda0p5",
            "original_writer_auc": (
                original_auc_2026
            ),
            "intervened_writer_auc": (
                intervened_auc_2026
            ),
            "residual_script_auc": (
                residual_script_auc_2026
            ),
            "mean_script_reliance": (
                pair_effect_2026_df[
                    "script_reliance"
                ].mean()
            ),
            "reliance_error_auc": (
                repeated_uncertainty_2026_df[
                    "reliance_auc"
                ].mean()
            ),
            "combined_auc_gain": (
                repeated_uncertainty_2026_df[
                    "combined_auc_gain"
                ].mean()
            ),
            "combined_auc_better": int(
                (
                    repeated_uncertainty_2026_df[
                        "combined_auc_gain"
                    ] > 0
                ).sum()
            ),
            "aurc_better": int(
                (
                    repeated_uncertainty_2026_df[
                        "aurc_improvement"
                    ] > 0
                ).sum()
            ),
        },
    ]
)

model_reliance_summary_df[
    "intervention_auc_change"
] = (
    model_reliance_summary_df[
        "intervened_writer_auc"
    ]
    - model_reliance_summary_df[
        "original_writer_auc"
    ]
)

print(
    model_reliance_summary_df[
        [
            "training_seed",
            "model",
            "original_writer_auc",
            "intervened_writer_auc",
            "intervention_auc_change",
            "residual_script_auc",
            "mean_script_reliance",
            "reliance_error_auc",
            "combined_auc_gain",
            "combined_auc_better",
            "aurc_better",
        ]
    ]
    .round(6)
    .to_string(
        index=False
    )
)

 training_seed               model  original_writer_auc  intervened_writer_auc  intervention_auc_change  residual_script_auc  mean_script_reliance  reliance_error_auc  combined_auc_gain  combined_auc_better  aurc_better
           123     lambda0_control             0.738789               0.768275                 0.029486             0.891103              0.063327            0.617219           0.020262                   35           37
           123 batch_alt_lambda0p5             0.776425               0.776584                 0.000159             0.825415              0.009798            0.592974           0.022980                   40           40
          2026     lambda0_control             0.738279               0.762910                 0.024631             0.926977              0.036477            0.476666          -0.003485                   10           14
          2026 batch_alt_lambda0p5             0.770031               0.770636                 0.000605             0.70

In [31]:
def summarize_family_intervention(
    pair_df,
    training_seed,
    model_name,
):
    rows = []

    for family, family_df in (
        pair_df.groupby(
            "family"
        )
    ):
        labels = family_df[
            "pair_label"
        ].to_numpy(
            dtype=np.int64
        )

        original_auc_family = roc_auc_score(
            labels,
            family_df[
                "original_score"
            ],
        )

        intervened_auc_family = roc_auc_score(
            labels,
            family_df[
                "intervened_score"
            ],
        )

        rows.append(
            {
                "training_seed": (
                    training_seed
                ),
                "model": (
                    model_name
                ),
                "family": (
                    family
                ),
                "original_auc": (
                    original_auc_family
                ),
                "intervened_auc": (
                    intervened_auc_family
                ),
                "auc_change": (
                    intervened_auc_family
                    - original_auc_family
                ),
                "mean_script_reliance": (
                    family_df[
                        "script_reliance"
                    ].mean()
                ),
                "median_script_reliance": (
                    family_df[
                        "script_reliance"
                    ].median()
                ),
            }
        )

    return pd.DataFrame(
        rows
    )


family_model_comparison_df = pd.concat(
    [
        summarize_family_intervention(
            seed123_control_pair_df,
            123,
            "lambda0_control",
        ),
        summarize_family_intervention(
            pair_effect_df,
            123,
            "batch_alt_lambda0p5",
        ),
        summarize_family_intervention(
            seed2026_control_pair_df,
            2026,
            "lambda0_control",
        ),
        summarize_family_intervention(
            pair_effect_2026_df,
            2026,
            "batch_alt_lambda0p5",
        ),
    ],
    ignore_index=True,
)

print(
    family_model_comparison_df
    .round(6)
    .to_string(
        index=False
    )
)

 training_seed               model        family  original_auc  intervened_auc  auc_change  mean_script_reliance  median_script_reliance
           123     lambda0_control  cross_script      0.721810        0.735458    0.013647              0.049595                0.041368
           123     lambda0_control within_script      0.847526        0.838765   -0.008760              0.090792                0.080464
           123 batch_alt_lambda0p5  cross_script      0.745549        0.744374   -0.001175              0.008887                0.006722
           123 batch_alt_lambda0p5 within_script      0.842680        0.839838   -0.002842              0.011618                0.007573
          2026     lambda0_control  cross_script      0.722413        0.732454    0.010041              0.033011                0.026576
          2026     lambda0_control within_script      0.842089        0.838554   -0.003535              0.043407                0.041173
          2026 batch_alt_lambda0p5  cross

In [32]:
paired_reliance_rows = []

for training_seed in [
    123,
    2026,
]:
    control_row = (
        family_model_comparison_df[
            (
                family_model_comparison_df[
                    "training_seed"
                ] == training_seed
            )
            & (
                family_model_comparison_df[
                    "model"
                ] == "lambda0_control"
            )
            & (
                family_model_comparison_df[
                    "family"
                ] == "cross_script"
            )
        ]
        .iloc[
            0
        ]
    )

    batch_row = (
        family_model_comparison_df[
            (
                family_model_comparison_df[
                    "training_seed"
                ] == training_seed
            )
            & (
                family_model_comparison_df[
                    "model"
                ] == "batch_alt_lambda0p5"
            )
            & (
                family_model_comparison_df[
                    "family"
                ] == "cross_script"
            )
        ]
        .iloc[
            0
        ]
    )

    reliance_reduction = (
        control_row[
            "mean_script_reliance"
        ]
        - batch_row[
            "mean_script_reliance"
        ]
    )

    relative_reliance_reduction = (
        reliance_reduction
        / control_row[
            "mean_script_reliance"
        ]
    )

    paired_reliance_rows.append(
        {
            "training_seed": (
                training_seed
            ),
            "control_cross_reliance": (
                control_row[
                    "mean_script_reliance"
                ]
            ),
            "batch_cross_reliance": (
                batch_row[
                    "mean_script_reliance"
                ]
            ),
            "absolute_reliance_reduction": (
                reliance_reduction
            ),
            "relative_reliance_reduction": (
                relative_reliance_reduction
            ),
            "control_intervention_auc_change": (
                control_row[
                    "auc_change"
                ]
            ),
            "batch_intervention_auc_change": (
                batch_row[
                    "auc_change"
                ]
            ),
        }
    )


paired_reliance_df = pd.DataFrame(
    paired_reliance_rows
)

print(
    paired_reliance_df
    .round(6)
    .to_string(
        index=False
    )
)

print()

print(
    "Mean relative cross-script reliance reduction (%):",
    100.0
    * paired_reliance_df[
        "relative_reliance_reduction"
    ].mean(),
)

print(
    "Control mean intervention AUC change:",
    paired_reliance_df[
        "control_intervention_auc_change"
    ].mean(),
)

print(
    "Batch-alt mean intervention AUC change:",
    paired_reliance_df[
        "batch_intervention_auc_change"
    ].mean(),
)

 training_seed  control_cross_reliance  batch_cross_reliance  absolute_reliance_reduction  relative_reliance_reduction  control_intervention_auc_change  batch_intervention_auc_change
           123                0.049595              0.008887                     0.040707                     0.820801                         0.013647                      -0.001175
          2026                0.033011              0.012144                     0.020867                     0.632118                         0.010041                      -0.001040

Mean relative cross-script reliance reduction (%): 72.64590822660124
Control mean intervention AUC change: 0.011843945674860912
Batch-alt mean intervention AUC change: -0.0011076465387292456


In [33]:
RANDOM_SUBSPACE_TRIALS = 20


def remove_subspace(
    embeddings,
    basis_matrix,
):
    projected = (
        embeddings
        - (
            embeddings
            @ basis_matrix.T
        )
        @ basis_matrix
    )

    projected = (
        projected
        / np.linalg.norm(
            projected,
            axis=1,
            keepdims=True,
        )
    )

    return projected


def evaluate_random_subspace_controls(
    checkpoint_path,
    training_seed,
    model_name,
):
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )

    model = (
        ScriptAdversarialWriterModel()
        .to(
            TRAIN_DEVICE
        )
    )

    model.load_state_dict(
        checkpoint[
            "model_state_dict"
        ],
        strict=True,
    )

    model.eval()

    validation_embeddings_current, _, validation_filenames_current = (
        extract_embeddings_and_labels(
            model,
            validation_loader,
        )
    )

    original_map = dict(
        zip(
            validation_filenames_current,
            validation_embeddings_current,
        )
    )

    original_a = np.stack(
        [
            original_map[
                filename
            ]
            for filename in (
                validation_pairs_df[
                    "filename_a"
                ]
            )
        ]
    )

    original_b = np.stack(
        [
            original_map[
                filename
            ]
            for filename in (
                validation_pairs_df[
                    "filename_b"
                ]
            )
        ]
    )

    original_scores_current = np.sum(
        original_a
        * original_b,
        axis=1,
    )

    page_a = validation_pairs_df[
        "filename_a"
    ].map(
        filename_page_id
    )

    page_b = validation_pairs_df[
        "filename_b"
    ].map(
        filename_page_id
    )

    conditions = [
        CONDITION_BY_PAGE_PAIR[
            (
                int(
                    first_page
                ),
                int(
                    second_page
                ),
            )
        ]
        for first_page, second_page in zip(
            page_a,
            page_b,
        )
    ]

    cross_mask = np.array(
        [
            condition.startswith(
                "cross_"
            )
            for condition in conditions
        ]
    )

    cross_labels = validation_pairs_df[
        "pair_label"
    ].to_numpy(
        dtype=np.int64
    )[
        cross_mask
    ]

    original_cross_scores = (
        original_scores_current[
            cross_mask
        ]
    )

    original_cross_auc = roc_auc_score(
        cross_labels,
        original_cross_scores,
    )

    rows = []

    for trial in range(
        RANDOM_SUBSPACE_TRIALS
    ):
        rng = np.random.default_rng(
            50_000
            + training_seed
            + trial
        )

        random_matrix = rng.normal(
            size=(
                512,
                SELECTED_SCRIPT_SUBSPACE_DIMENSION,
            )
        )

        random_basis, _ = np.linalg.qr(
            random_matrix
        )

        random_basis = (
            random_basis.T
        )

        intervened_embeddings = (
            remove_subspace(
                validation_embeddings_current,
                random_basis,
            )
        )

        intervened_map = dict(
            zip(
                validation_filenames_current,
                intervened_embeddings,
            )
        )

        intervened_a = np.stack(
            [
                intervened_map[
                    filename
                ]
                for filename in (
                    validation_pairs_df[
                        "filename_a"
                    ]
                )
            ]
        )

        intervened_b = np.stack(
            [
                intervened_map[
                    filename
                ]
                for filename in (
                    validation_pairs_df[
                        "filename_b"
                    ]
                )
            ]
        )

        intervened_scores = np.sum(
            intervened_a
            * intervened_b,
            axis=1,
        )

        random_reliance = np.abs(
            intervened_scores
            - original_scores_current
        )

        intervened_cross_auc = (
            roc_auc_score(
                cross_labels,
                intervened_scores[
                    cross_mask
                ],
            )
        )

        rows.append(
            {
                "training_seed": (
                    training_seed
                ),
                "model": (
                    model_name
                ),
                "trial": (
                    trial
                ),
                "random_cross_reliance": float(
                    random_reliance[
                        cross_mask
                    ].mean()
                ),
                "random_cross_auc_change": float(
                    intervened_cross_auc
                    - original_cross_auc
                ),
            }
        )

    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return pd.DataFrame(
        rows
    )


random_control_df = pd.concat(
    [
        evaluate_random_subspace_controls(
            seed123_control_checkpoint_path,
            123,
            "lambda0_control",
        ),
        evaluate_random_subspace_controls(
            seed123_checkpoint_path,
            123,
            "batch_alt_lambda0p5",
        ),
        evaluate_random_subspace_controls(
            seed2026_control_checkpoint_path,
            2026,
            "lambda0_control",
        ),
        evaluate_random_subspace_controls(
            seed2026_checkpoint_path,
            2026,
            "batch_alt_lambda0p5",
        ),
    ],
    ignore_index=True,
)

print(
    "Random-subspace evaluations:",
    len(
        random_control_df
    ),
)

print(
    random_control_df.groupby(
        [
            "training_seed",
            "model",
        ]
    )[
        [
            "random_cross_reliance",
            "random_cross_auc_change",
        ]
    ]
    .mean()
    .round(6)
    .to_string()
)

Random-subspace evaluations: 80
                                   random_cross_reliance  random_cross_auc_change
training_seed model                                                              
123           batch_alt_lambda0p5               0.012594                -0.001652
              lambda0_control                   0.014376                -0.002130
2026          batch_alt_lambda0p5               0.013601                -0.001102
              lambda0_control                   0.013760                -0.002119


In [34]:
script_specificity_rows = []

for training_seed in [
    123,
    2026,
]:
    for model_name in [
        "lambda0_control",
        "batch_alt_lambda0p5",
    ]:
        script_row = (
            family_model_comparison_df[
                (
                    family_model_comparison_df[
                        "training_seed"
                    ] == training_seed
                )
                & (
                    family_model_comparison_df[
                        "model"
                    ] == model_name
                )
                & (
                    family_model_comparison_df[
                        "family"
                    ] == "cross_script"
                )
            ]
            .iloc[
                0
            ]
        )

        random_rows = (
            random_control_df[
                (
                    random_control_df[
                        "training_seed"
                    ] == training_seed
                )
                & (
                    random_control_df[
                        "model"
                    ] == model_name
                )
            ]
        )

        random_reliance_mean = (
            random_rows[
                "random_cross_reliance"
            ].mean()
        )

        random_reliance_std = (
            random_rows[
                "random_cross_reliance"
            ].std(
                ddof=1
            )
        )

        random_auc_change_mean = (
            random_rows[
                "random_cross_auc_change"
            ].mean()
        )

        random_auc_change_std = (
            random_rows[
                "random_cross_auc_change"
            ].std(
                ddof=1
            )
        )

        script_specificity_rows.append(
            {
                "training_seed": (
                    training_seed
                ),
                "model": (
                    model_name
                ),
                "script_cross_reliance": (
                    script_row[
                        "mean_script_reliance"
                    ]
                ),
                "random_cross_reliance_mean": (
                    random_reliance_mean
                ),
                "random_cross_reliance_std": (
                    random_reliance_std
                ),
                "script_vs_random_reliance_ratio": (
                    script_row[
                        "mean_script_reliance"
                    ]
                    / random_reliance_mean
                ),
                "script_cross_auc_change": (
                    script_row[
                        "auc_change"
                    ]
                ),
                "random_cross_auc_change_mean": (
                    random_auc_change_mean
                ),
                "random_cross_auc_change_std": (
                    random_auc_change_std
                ),
            }
        )


script_specificity_df = pd.DataFrame(
    script_specificity_rows
)

print(
    script_specificity_df
    .round(6)
    .to_string(
        index=False
    )
)

 training_seed               model  script_cross_reliance  random_cross_reliance_mean  random_cross_reliance_std  script_vs_random_reliance_ratio  script_cross_auc_change  random_cross_auc_change_mean  random_cross_auc_change_std
           123     lambda0_control               0.049595                    0.014376                   0.002741                         3.449840                 0.013647                     -0.002130                     0.003497
           123 batch_alt_lambda0p5               0.008887                    0.012594                   0.002754                         0.705652                -0.001175                     -0.001652                     0.002217
          2026     lambda0_control               0.033011                    0.013760                   0.004528                         2.399123                 0.010041                     -0.002119                     0.002735
          2026 batch_alt_lambda0p5               0.012144                    0.0

In [35]:
validation_filename_index = {
    filename: index
    for index, filename in enumerate(
        validation_filenames
    )
}

pair_indices_a = np.array(
    [
        validation_filename_index[
            filename
        ]
        for filename in (
            validation_pairs_df[
                "filename_a"
            ]
        )
    ]
)

pair_indices_b = np.array(
    [
        validation_filename_index[
            filename
        ]
        for filename in (
            validation_pairs_df[
                "filename_b"
            ]
        )
    ]
)


def calculate_mean_random_reliance(
    embeddings,
    original_pair_scores,
    training_seed,
):
    trial_reliance = []

    for trial in range(
        RANDOM_SUBSPACE_TRIALS
    ):
        rng = np.random.default_rng(
            50_000
            + training_seed
            + trial
        )

        random_matrix = rng.normal(
            size=(
                512,
                SELECTED_SCRIPT_SUBSPACE_DIMENSION,
            )
        )

        random_basis, _ = np.linalg.qr(
            random_matrix
        )

        random_basis = (
            random_basis.T
        )

        random_intervened_embeddings = (
            remove_subspace(
                embeddings,
                random_basis,
            )
        )

        random_scores = np.sum(
            random_intervened_embeddings[
                pair_indices_a
            ]
            * random_intervened_embeddings[
                pair_indices_b
            ],
            axis=1,
        )

        trial_reliance.append(
            np.abs(
                random_scores
                - original_pair_scores
            )
        )

    trial_reliance = np.stack(
        trial_reliance,
        axis=0,
    )

    return (
        trial_reliance.mean(
            axis=0
        ),
        trial_reliance.std(
            axis=0,
        ),
    )


random_reliance_mean_123, random_reliance_std_123 = (
    calculate_mean_random_reliance(
        validation_embeddings,
        original_scores,
        123,
    )
)

random_reliance_mean_2026, random_reliance_std_2026 = (
    calculate_mean_random_reliance(
        validation_embeddings_2026,
        original_scores_2026,
        2026,
    )
)

pair_effect_df[
    "random_reliance"
] = random_reliance_mean_123

pair_effect_2026_df[
    "random_reliance"
] = random_reliance_mean_2026

print(
    "Seed-123 mean script reliance:",
    pair_effect_df[
        "script_reliance"
    ].mean(),
)

print(
    "Seed-123 mean random reliance:",
    pair_effect_df[
        "random_reliance"
    ].mean(),
)

print()

print(
    "Seed-2026 mean script reliance:",
    pair_effect_2026_df[
        "script_reliance"
    ].mean(),
)

print(
    "Seed-2026 mean random reliance:",
    pair_effect_2026_df[
        "random_reliance"
    ].mean(),
)

Seed-123 mean script reliance: 0.009797601673933388
Seed-123 mean random reliance: 0.012535321428017863

Seed-2026 mean script reliance: 0.010207820265921593
Seed-2026 mean random reliance: 0.013451642640857502


In [36]:
def evaluate_repeated_feature_uncertainty(
    cross_pairs_df,
    feature_column,
):
    rows = []

    for split_index in range(
        REPEATED_WRITER_SPLITS
    ):
        rng = np.random.default_rng(
            1000
            + SPLIT_SEED
            + split_index
        )

        shuffled_writers = rng.permutation(
            validation_writers
        )

        writer_half_a = shuffled_writers[
            :28
        ]

        writer_half_b = shuffled_writers[
            28:
        ]

        for fold, calibration_writers, evaluation_writers_current in [
            (
                "A_to_B",
                writer_half_a,
                writer_half_b,
            ),
            (
                "B_to_A",
                writer_half_b,
                writer_half_a,
            ),
        ]:
            calibration_mask = (
                cross_pairs_df[
                    "writer_a"
                ].isin(
                    calibration_writers
                )
                & cross_pairs_df[
                    "writer_b"
                ].isin(
                    calibration_writers
                )
            )

            evaluation_mask = (
                cross_pairs_df[
                    "writer_a"
                ].isin(
                    evaluation_writers_current
                )
                & cross_pairs_df[
                    "writer_b"
                ].isin(
                    evaluation_writers_current
                )
            )

            calibration_df = (
                cross_pairs_df[
                    calibration_mask
                ]
                .copy()
                .reset_index(drop=True)
            )

            evaluation_df = (
                cross_pairs_df[
                    evaluation_mask
                ]
                .copy()
                .reset_index(drop=True)
            )

            _, threshold = (
                calculate_interpolated_eer(
                    calibration_df[
                        "pair_label"
                    ].to_numpy(
                        dtype=np.int64
                    ),
                    calibration_df[
                        "original_score"
                    ].to_numpy(),
                )
            )

            for dataframe in [
                calibration_df,
                evaluation_df,
            ]:
                dataframe[
                    "verification_error"
                ] = (
                    (
                        dataframe[
                            "original_score"
                        ]
                        >= threshold
                    ).astype(
                        np.int64
                    )
                    != dataframe[
                        "pair_label"
                    ].to_numpy(
                        dtype=np.int64
                    )
                ).astype(
                    np.int64
                )

                dataframe[
                    "margin_uncertainty"
                ] = (
                    -np.abs(
                        dataframe[
                            "original_score"
                        ]
                        - threshold
                    )
                )

            calibration_errors = (
                calibration_df[
                    "verification_error"
                ].to_numpy(
                    dtype=np.int64
                )
            )

            evaluation_errors = (
                evaluation_df[
                    "verification_error"
                ].to_numpy(
                    dtype=np.int64
                )
            )

            margin_model = (
                build_error_predictor()
            )

            combined_model = (
                build_error_predictor()
            )

            margin_model.fit(
                calibration_df[
                    [
                        "margin_uncertainty"
                    ]
                ].to_numpy(),
                calibration_errors,
            )

            combined_model.fit(
                calibration_df[
                    [
                        "margin_uncertainty",
                        feature_column,
                    ]
                ].to_numpy(),
                calibration_errors,
            )

            margin_risk = (
                margin_model.predict_proba(
                    evaluation_df[
                        [
                            "margin_uncertainty"
                        ]
                    ].to_numpy()
                )[
                    :,
                    1
                ]
            )

            combined_risk = (
                combined_model.predict_proba(
                    evaluation_df[
                        [
                            "margin_uncertainty",
                            feature_column,
                        ]
                    ].to_numpy()
                )[
                    :,
                    1
                ]
            )

            margin_auc = roc_auc_score(
                evaluation_errors,
                margin_risk,
            )

            combined_auc = roc_auc_score(
                evaluation_errors,
                combined_risk,
            )

            margin_aurc = calculate_aurc(
                evaluation_errors,
                margin_risk,
            )

            combined_aurc = calculate_aurc(
                evaluation_errors,
                combined_risk,
            )

            rows.append(
                {
                    "split_index": (
                        split_index
                    ),
                    "fold": (
                        fold
                    ),
                    "combined_auc_gain": (
                        combined_auc
                        - margin_auc
                    ),
                    "aurc_improvement": (
                        margin_aurc
                        - combined_aurc
                    ),
                }
            )

    return pd.DataFrame(
        rows
    )


batch123_cross_with_random_df = (
    pair_effect_df[
        pair_effect_df[
            "family"
        ] == "cross_script"
    ]
    .copy()
    .reset_index(drop=True)
)

batch2026_cross_with_random_df = (
    pair_effect_2026_df[
        pair_effect_2026_df[
            "family"
        ] == "cross_script"
    ]
    .copy()
    .reset_index(drop=True)
)

random_uncertainty_123_df = (
    evaluate_repeated_feature_uncertainty(
        batch123_cross_with_random_df,
        "random_reliance",
    )
)

random_uncertainty_2026_df = (
    evaluate_repeated_feature_uncertainty(
        batch2026_cross_with_random_df,
        "random_reliance",
    )
)

print(
    "Seed 123 script-reliance mean AUC gain:",
    repeated_uncertainty_df[
        "combined_auc_gain"
    ].mean(),
)

print(
    "Seed 123 random-reliance mean AUC gain:",
    random_uncertainty_123_df[
        "combined_auc_gain"
    ].mean(),
)

print(
    "Seed 123 script better than random:",
    int(
        (
            repeated_uncertainty_df[
                "combined_auc_gain"
            ].to_numpy()
            > random_uncertainty_123_df[
                "combined_auc_gain"
            ].to_numpy()
        ).sum()
    ),
    "/ 40",
)

print()

print(
    "Seed 2026 script-reliance mean AUC gain:",
    repeated_uncertainty_2026_df[
        "combined_auc_gain"
    ].mean(),
)

print(
    "Seed 2026 random-reliance mean AUC gain:",
    random_uncertainty_2026_df[
        "combined_auc_gain"
    ].mean(),
)

print(
    "Seed 2026 script better than random:",
    int(
        (
            repeated_uncertainty_2026_df[
                "combined_auc_gain"
            ].to_numpy()
            > random_uncertainty_2026_df[
                "combined_auc_gain"
            ].to_numpy()
        ).sum()
    ),
    "/ 40",
)

print()

print(
    "Seed 123 script AURC improvement:",
    repeated_uncertainty_df[
        "aurc_improvement"
    ].mean(),
)

print(
    "Seed 123 random AURC improvement:",
    random_uncertainty_123_df[
        "aurc_improvement"
    ].mean(),
)

print()

print(
    "Seed 2026 script AURC improvement:",
    repeated_uncertainty_2026_df[
        "aurc_improvement"
    ].mean(),
)

print(
    "Seed 2026 random AURC improvement:",
    random_uncertainty_2026_df[
        "aurc_improvement"
    ].mean(),
)

Seed 123 script-reliance mean AUC gain: 0.022980119601306116
Seed 123 random-reliance mean AUC gain: 0.19213490079856627
Seed 123 script better than random: 0 / 40

Seed 2026 script-reliance mean AUC gain: 0.017592438669223772
Seed 2026 random-reliance mean AUC gain: 0.21967334035767383
Seed 2026 script better than random: 0 / 40

Seed 123 script AURC improvement: 0.008950803573481391
Seed 123 random AURC improvement: 0.07367841899735408

Seed 2026 script AURC improvement: 0.007645061116021985
Seed 2026 random AURC improvement: 0.08546822765393951


In [37]:
def reliance_correlation_summary(
    pair_df,
    threshold,
    training_seed,
):
    cross_df = (
        pair_df[
            pair_df[
                "family"
            ] == "cross_script"
        ]
        .copy()
        .reset_index(drop=True)
    )

    cross_df[
        "signed_margin"
    ] = (
        cross_df[
            "original_score"
        ]
        - threshold
    )

    cross_df[
        "absolute_margin"
    ] = np.abs(
        cross_df[
            "signed_margin"
        ]
    )

    correlation_df = cross_df[
        [
            "original_score",
            "signed_margin",
            "absolute_margin",
            "script_reliance",
            "random_reliance",
        ]
    ]

    rank_correlation_df = (
        correlation_df
        .rank()
        .corr()
    )

    print(
        "Training seed:",
        training_seed,
    )

    print(
        rank_correlation_df[
            [
                "script_reliance",
                "random_reliance",
            ]
        ]
        .round(6)
        .to_string()
    )

    print()


reliance_correlation_summary(
    pair_effect_df,
    float(
        seed123_checkpoint[
            "validation_eer_threshold"
        ]
    ),
    123,
)

reliance_correlation_summary(
    pair_effect_2026_df,
    float(
        seed2026_checkpoint[
            "validation_eer_threshold"
        ]
    ),
    2026,
)

Training seed: 123
                 script_reliance  random_reliance
original_score          0.125346        -0.861172
signed_margin           0.125346        -0.861172
absolute_margin        -0.062014         0.615857
script_reliance         1.000000        -0.129854
random_reliance        -0.129854         1.000000

Training seed: 2026
                 script_reliance  random_reliance
original_score          0.371118        -0.889007
signed_margin           0.371118        -0.889007
absolute_margin        -0.350734         0.620248
script_reliance         1.000000        -0.330942
random_reliance        -0.330942         1.000000



In [38]:
def evaluate_strong_uncertainty_baselines(
    cross_pairs_df,
):
    rows = []

    for split_index in range(
        REPEATED_WRITER_SPLITS
    ):
        rng = np.random.default_rng(
            1000
            + SPLIT_SEED
            + split_index
        )

        shuffled_writers = rng.permutation(
            validation_writers
        )

        writer_half_a = shuffled_writers[
            :28
        ]

        writer_half_b = shuffled_writers[
            28:
        ]

        for fold, calibration_writers, evaluation_writers_current in [
            (
                "A_to_B",
                writer_half_a,
                writer_half_b,
            ),
            (
                "B_to_A",
                writer_half_b,
                writer_half_a,
            ),
        ]:
            calibration_mask = (
                cross_pairs_df[
                    "writer_a"
                ].isin(
                    calibration_writers
                )
                & cross_pairs_df[
                    "writer_b"
                ].isin(
                    calibration_writers
                )
            )

            evaluation_mask = (
                cross_pairs_df[
                    "writer_a"
                ].isin(
                    evaluation_writers_current
                )
                & cross_pairs_df[
                    "writer_b"
                ].isin(
                    evaluation_writers_current
                )
            )

            calibration_df = (
                cross_pairs_df[
                    calibration_mask
                ]
                .copy()
                .reset_index(drop=True)
            )

            evaluation_df = (
                cross_pairs_df[
                    evaluation_mask
                ]
                .copy()
                .reset_index(drop=True)
            )

            _, threshold = (
                calculate_interpolated_eer(
                    calibration_df[
                        "pair_label"
                    ].to_numpy(
                        dtype=np.int64
                    ),
                    calibration_df[
                        "original_score"
                    ].to_numpy(),
                )
            )

            for dataframe in [
                calibration_df,
                evaluation_df,
            ]:
                dataframe[
                    "verification_error"
                ] = (
                    (
                        dataframe[
                            "original_score"
                        ]
                        >= threshold
                    ).astype(
                        np.int64
                    )
                    != dataframe[
                        "pair_label"
                    ].to_numpy(
                        dtype=np.int64
                    )
                ).astype(
                    np.int64
                )

                dataframe[
                    "signed_margin"
                ] = (
                    dataframe[
                        "original_score"
                    ]
                    - threshold
                )

                dataframe[
                    "absolute_margin"
                ] = np.abs(
                    dataframe[
                        "signed_margin"
                    ]
                )

            calibration_errors = (
                calibration_df[
                    "verification_error"
                ].to_numpy(
                    dtype=np.int64
                )
            )

            evaluation_errors = (
                evaluation_df[
                    "verification_error"
                ].to_numpy(
                    dtype=np.int64
                )
            )

            score_model = (
                build_error_predictor()
            )

            script_model = (
                build_error_predictor()
            )

            random_model = (
                build_error_predictor()
            )

            score_model.fit(
                calibration_df[
                    [
                        "signed_margin",
                        "absolute_margin",
                    ]
                ].to_numpy(),
                calibration_errors,
            )

            script_model.fit(
                calibration_df[
                    [
                        "signed_margin",
                        "absolute_margin",
                        "script_reliance",
                    ]
                ].to_numpy(),
                calibration_errors,
            )

            random_model.fit(
                calibration_df[
                    [
                        "signed_margin",
                        "absolute_margin",
                        "random_reliance",
                    ]
                ].to_numpy(),
                calibration_errors,
            )

            score_risk = (
                score_model.predict_proba(
                    evaluation_df[
                        [
                            "signed_margin",
                            "absolute_margin",
                        ]
                    ].to_numpy()
                )[
                    :,
                    1
                ]
            )

            script_risk = (
                script_model.predict_proba(
                    evaluation_df[
                        [
                            "signed_margin",
                            "absolute_margin",
                            "script_reliance",
                        ]
                    ].to_numpy()
                )[
                    :,
                    1
                ]
            )

            random_risk = (
                random_model.predict_proba(
                    evaluation_df[
                        [
                            "signed_margin",
                            "absolute_margin",
                            "random_reliance",
                        ]
                    ].to_numpy()
                )[
                    :,
                    1
                ]
            )

            score_auc = roc_auc_score(
                evaluation_errors,
                score_risk,
            )

            script_auc = roc_auc_score(
                evaluation_errors,
                script_risk,
            )

            random_auc = roc_auc_score(
                evaluation_errors,
                random_risk,
            )

            score_aurc = calculate_aurc(
                evaluation_errors,
                score_risk,
            )

            script_aurc = calculate_aurc(
                evaluation_errors,
                script_risk,
            )

            random_aurc = calculate_aurc(
                evaluation_errors,
                random_risk,
            )

            rows.append(
                {
                    "split_index": (
                        split_index
                    ),
                    "fold": (
                        fold
                    ),
                    "score_auc": (
                        score_auc
                    ),
                    "script_auc": (
                        script_auc
                    ),
                    "random_auc": (
                        random_auc
                    ),
                    "script_auc_gain": (
                        script_auc
                        - score_auc
                    ),
                    "random_auc_gain": (
                        random_auc
                        - score_auc
                    ),
                    "score_aurc": (
                        score_aurc
                    ),
                    "script_aurc": (
                        script_aurc
                    ),
                    "random_aurc": (
                        random_aurc
                    ),
                    "script_aurc_improvement": (
                        score_aurc
                        - script_aurc
                    ),
                    "random_aurc_improvement": (
                        score_aurc
                        - random_aurc
                    ),
                }
            )

    return pd.DataFrame(
        rows
    )


strong_uncertainty_123_df = (
    evaluate_strong_uncertainty_baselines(
        batch123_cross_with_random_df
    )
)

strong_uncertainty_2026_df = (
    evaluate_strong_uncertainty_baselines(
        batch2026_cross_with_random_df
    )
)

for training_seed, result_df in [
    (
        123,
        strong_uncertainty_123_df,
    ),
    (
        2026,
        strong_uncertainty_2026_df,
    ),
]:
    print(
        "Training seed:",
        training_seed,
    )

    print(
        "Strong score-only AUC:",
        result_df[
            "score_auc"
        ].mean(),
    )

    print(
        "Score + script AUC:",
        result_df[
            "script_auc"
        ].mean(),
    )

    print(
        "Score + random AUC:",
        result_df[
            "random_auc"
        ].mean(),
    )

    print(
        "Script gain over strong baseline:",
        result_df[
            "script_auc_gain"
        ].mean(),
    )

    print(
        "Random gain over strong baseline:",
        result_df[
            "random_auc_gain"
        ].mean(),
    )

    print(
        "Script AUC better:",
        int(
            (
                result_df[
                    "script_auc_gain"
                ] > 0
            ).sum()
        ),
        "/ 40",
    )

    print(
        "Random AUC better:",
        int(
            (
                result_df[
                    "random_auc_gain"
                ] > 0
            ).sum()
        ),
        "/ 40",
    )

    print(
        "Script AURC improvement:",
        result_df[
            "script_aurc_improvement"
        ].mean(),
    )

    print(
        "Random AURC improvement:",
        result_df[
            "random_aurc_improvement"
        ].mean(),
    )

    print()

Training seed: 123
Strong score-only AUC: 0.9655580185686052
Score + script AUC: 0.9655052252484481
Score + random AUC: 0.9654886868280798
Script gain over strong baseline: -5.279332015701854e-05
Random gain over strong baseline: -6.933174052539403e-05
Script AUC better: 11 / 40
Random AUC better: 15 / 40
Script AURC improvement: -1.6551195190405744e-05
Random AURC improvement: -8.444376829487393e-06

Training seed: 2026
Strong score-only AUC: 0.965749095990026
Score + script AUC: 0.9657124830303339
Score + random AUC: 0.9656296744632591
Script gain over strong baseline: -3.661295969227818e-05
Random gain over strong baseline: -0.00011942152676676087
Script AUC better: 9 / 40
Random AUC better: 10 / 40
Script AURC improvement: -1.2944666447092974e-05
Random AURC improvement: -4.598993103092294e-05



In [39]:
def evaluate_decision_side_uncertainty(
    cross_pairs_df,
):
    rows = []

    for split_index in range(
        REPEATED_WRITER_SPLITS
    ):
        rng = np.random.default_rng(
            1000
            + SPLIT_SEED
            + split_index
        )

        shuffled_writers = rng.permutation(
            validation_writers
        )

        writer_half_a = shuffled_writers[
            :28
        ]

        writer_half_b = shuffled_writers[
            28:
        ]

        for fold, calibration_writers, evaluation_writers_current in [
            (
                "A_to_B",
                writer_half_a,
                writer_half_b,
            ),
            (
                "B_to_A",
                writer_half_b,
                writer_half_a,
            ),
        ]:
            calibration_mask = (
                cross_pairs_df[
                    "writer_a"
                ].isin(
                    calibration_writers
                )
                & cross_pairs_df[
                    "writer_b"
                ].isin(
                    calibration_writers
                )
            )

            evaluation_mask = (
                cross_pairs_df[
                    "writer_a"
                ].isin(
                    evaluation_writers_current
                )
                & cross_pairs_df[
                    "writer_b"
                ].isin(
                    evaluation_writers_current
                )
            )

            calibration_df = (
                cross_pairs_df[
                    calibration_mask
                ]
                .copy()
                .reset_index(drop=True)
            )

            evaluation_df = (
                cross_pairs_df[
                    evaluation_mask
                ]
                .copy()
                .reset_index(drop=True)
            )

            _, threshold = (
                calculate_interpolated_eer(
                    calibration_df[
                        "pair_label"
                    ].to_numpy(
                        dtype=np.int64
                    ),
                    calibration_df[
                        "original_score"
                    ].to_numpy(),
                )
            )

            for dataframe in [
                calibration_df,
                evaluation_df,
            ]:
                dataframe[
                    "prediction"
                ] = (
                    dataframe[
                        "original_score"
                    ]
                    >= threshold
                ).astype(
                    np.int64
                )

                dataframe[
                    "verification_error"
                ] = (
                    dataframe[
                        "prediction"
                    ]
                    != dataframe[
                        "pair_label"
                    ]
                ).astype(
                    np.int64
                )

                dataframe[
                    "score_distance"
                ] = np.abs(
                    dataframe[
                        "original_score"
                    ]
                    - threshold
                )

            for decision_name, decision_value in [
                (
                    "predicted_same",
                    1,
                ),
                (
                    "predicted_different",
                    0,
                ),
            ]:
                calibration_side = (
                    calibration_df[
                        calibration_df[
                            "prediction"
                        ] == decision_value
                    ]
                    .copy()
                    .reset_index(drop=True)
                )

                evaluation_side = (
                    evaluation_df[
                        evaluation_df[
                            "prediction"
                        ] == decision_value
                    ]
                    .copy()
                    .reset_index(drop=True)
                )

                calibration_errors = (
                    calibration_side[
                        "verification_error"
                    ].to_numpy(
                        dtype=np.int64
                    )
                )

                evaluation_errors = (
                    evaluation_side[
                        "verification_error"
                    ].to_numpy(
                        dtype=np.int64
                    )
                )

                if (
                    np.unique(
                        calibration_errors
                    ).size < 2
                    or np.unique(
                        evaluation_errors
                    ).size < 2
                ):
                    continue

                score_model = (
                    build_error_predictor()
                )

                script_model = (
                    build_error_predictor()
                )

                score_model.fit(
                    calibration_side[
                        [
                            "score_distance"
                        ]
                    ].to_numpy(),
                    calibration_errors,
                )

                script_model.fit(
                    calibration_side[
                        [
                            "score_distance",
                            "script_reliance",
                        ]
                    ].to_numpy(),
                    calibration_errors,
                )

                score_risk = (
                    score_model.predict_proba(
                        evaluation_side[
                            [
                                "score_distance"
                            ]
                        ].to_numpy()
                    )[
                        :,
                        1
                    ]
                )

                script_risk = (
                    script_model.predict_proba(
                        evaluation_side[
                            [
                                "score_distance",
                                "script_reliance",
                            ]
                        ].to_numpy()
                    )[
                        :,
                        1
                    ]
                )

                score_auc = roc_auc_score(
                    evaluation_errors,
                    score_risk,
                )

                script_auc = roc_auc_score(
                    evaluation_errors,
                    script_risk,
                )

                score_aurc = calculate_aurc(
                    evaluation_errors,
                    score_risk,
                )

                script_aurc = calculate_aurc(
                    evaluation_errors,
                    script_risk,
                )

                rows.append(
                    {
                        "split_index": (
                            split_index
                        ),
                        "fold": (
                            fold
                        ),
                        "decision": (
                            decision_name
                        ),
                        "pairs": (
                            len(
                                evaluation_side
                            )
                        ),
                        "error_rate": (
                            evaluation_errors.mean()
                        ),
                        "score_auc": (
                            score_auc
                        ),
                        "script_auc": (
                            script_auc
                        ),
                        "script_auc_gain": (
                            script_auc
                            - score_auc
                        ),
                        "score_aurc": (
                            score_aurc
                        ),
                        "script_aurc": (
                            script_aurc
                        ),
                        "script_aurc_improvement": (
                            score_aurc
                            - script_aurc
                        ),
                    }
                )

    return pd.DataFrame(
        rows
    )


decision_side_123_df = (
    evaluate_decision_side_uncertainty(
        batch123_cross_with_random_df
    )
)

decision_side_2026_df = (
    evaluate_decision_side_uncertainty(
        batch2026_cross_with_random_df
    )
)

print(
    "Seed-123 evaluations:",
    len(
        decision_side_123_df
    ),
)

print(
    "Seed-2026 evaluations:",
    len(
        decision_side_2026_df
    ),
)

Seed-123 evaluations: 80
Seed-2026 evaluations: 80


In [40]:
for training_seed, result_df in [
    (
        123,
        decision_side_123_df,
    ),
    (
        2026,
        decision_side_2026_df,
    ),
]:
    print(
        "Training seed:",
        training_seed,
    )

    for decision in [
        "predicted_same",
        "predicted_different",
    ]:
        decision_df = (
            result_df[
                result_df[
                    "decision"
                ] == decision
            ]
        )

        print()

        print(
            "Decision:",
            decision,
        )

        print(
            "Evaluations:",
            len(
                decision_df
            ),
        )

        print(
            "Mean score-only error AUC:",
            decision_df[
                "score_auc"
            ].mean(),
        )

        print(
            "Mean score + script AUC:",
            decision_df[
                "script_auc"
            ].mean(),
        )

        print(
            "Mean script AUC gain:",
            decision_df[
                "script_auc_gain"
            ].mean(),
        )

        print(
            "Script AUC better:",
            int(
                (
                    decision_df[
                        "script_auc_gain"
                    ] > 0
                ).sum()
            ),
            "/",
            len(
                decision_df
            ),
        )

        print(
            "Mean script AURC improvement:",
            decision_df[
                "script_aurc_improvement"
            ].mean(),
        )

        print(
            "Script AURC better:",
            int(
                (
                    decision_df[
                        "script_aurc_improvement"
                    ] > 0
                ).sum()
            ),
            "/",
            len(
                decision_df
            ),
        )

    print()

Training seed: 123

Decision: predicted_same
Evaluations: 40
Mean score-only error AUC: 0.6251027772565365
Mean score + script AUC: 0.6218702920231164
Mean script AUC gain: -0.003232485233420121
Script AUC better: 12 / 40
Mean script AURC improvement: 0.00011257054719213678
Script AURC better: 25 / 40

Decision: predicted_different
Evaluations: 40
Mean score-only error AUC: 0.6819783149823889
Mean score + script AUC: 0.679505846681786
Mean script AUC gain: -0.002472468300602981
Script AUC better: 24 / 40
Mean script AURC improvement: -0.00020049628720485426
Script AURC better: 23 / 40

Training seed: 2026

Decision: predicted_same
Evaluations: 40
Mean score-only error AUC: 0.6073200980032525
Mean score + script AUC: 0.6020817495892846
Mean script AUC gain: -0.005238348413967758
Script AUC better: 3 / 40
Mean script AURC improvement: -0.0004099570891536036
Script AURC better: 19 / 40

Decision: predicted_different
Evaluations: 40
Mean score-only error AUC: 0.6693185629124677
Mean score 

In [41]:
inner_removal_path = (
    REPORT_DIR
    / "script_subspace_removal_trajectory.csv"
)

model_summary_path = (
    REPORT_DIR
    / "model_reliance_summary.csv"
)

family_comparison_path = (
    REPORT_DIR
    / "family_model_comparison.csv"
)

paired_reliance_path = (
    REPORT_DIR
    / "paired_cross_script_reliance.csv"
)

random_control_path = (
    REPORT_DIR
    / "random_subspace_control.csv"
)

script_specificity_path = (
    REPORT_DIR
    / "script_subspace_specificity.csv"
)

strong_seed123_path = (
    REPORT_DIR
    / "strong_uncertainty_baseline_seed123.csv"
)

strong_seed2026_path = (
    REPORT_DIR
    / "strong_uncertainty_baseline_seed2026.csv"
)

decision_seed123_path = (
    REPORT_DIR
    / "decision_side_uncertainty_seed123.csv"
)

decision_seed2026_path = (
    REPORT_DIR
    / "decision_side_uncertainty_seed2026.csv"
)


inner_removal_df.to_csv(
    inner_removal_path,
    index=False,
)

model_reliance_summary_df.to_csv(
    model_summary_path,
    index=False,
)

family_model_comparison_df.to_csv(
    family_comparison_path,
    index=False,
)

paired_reliance_df.to_csv(
    paired_reliance_path,
    index=False,
)

random_control_df.to_csv(
    random_control_path,
    index=False,
)

script_specificity_df.to_csv(
    script_specificity_path,
    index=False,
)

strong_uncertainty_123_df.to_csv(
    strong_seed123_path,
    index=False,
)

strong_uncertainty_2026_df.to_csv(
    strong_seed2026_path,
    index=False,
)

decision_side_123_df.to_csv(
    decision_seed123_path,
    index=False,
)

decision_side_2026_df.to_csv(
    decision_seed2026_path,
    index=False,
)


saved_report_paths = [
    inner_removal_path,
    model_summary_path,
    family_comparison_path,
    paired_reliance_path,
    random_control_path,
    script_specificity_path,
    strong_seed123_path,
    strong_seed2026_path,
    decision_seed123_path,
    decision_seed2026_path,
]


print(
    "Saved reports:"
)

for path in saved_report_paths:
    print(
        "-",
        path.relative_to(
            PROJECT_ROOT
        ),
    )


assert all(
    path.exists()
    for path in saved_report_paths
)

Saved reports:
- reports/script_reliance_intervention/script_subspace_removal_trajectory.csv
- reports/script_reliance_intervention/model_reliance_summary.csv
- reports/script_reliance_intervention/family_model_comparison.csv
- reports/script_reliance_intervention/paired_cross_script_reliance.csv
- reports/script_reliance_intervention/random_subspace_control.csv
- reports/script_reliance_intervention/script_subspace_specificity.csv
- reports/script_reliance_intervention/strong_uncertainty_baseline_seed123.csv
- reports/script_reliance_intervention/strong_uncertainty_baseline_seed2026.csv
- reports/script_reliance_intervention/decision_side_uncertainty_seed123.csv
- reports/script_reliance_intervention/decision_side_uncertainty_seed2026.csv


In [42]:
notebook18_summary = {
    "experiment": (
        "Script-Reliance Intervention for "
        "Cross-Script Writer Verification"
    ),
    "selected_script_subspace_dimension": (
        SELECTED_SCRIPT_SUBSPACE_DIMENSION
    ),
    "selection_protocol": (
        "Development-only writer-disjoint "
        "subspace-train and monitor split"
    ),
    "batch_alt_seed123": {
        "original_writer_auc": float(
            original_auc
        ),
        "intervened_writer_auc": float(
            intervened_auc
        ),
        "residual_script_auc": float(
            final_validation_script_auc
        ),
        "mean_script_reliance": float(
            pair_effect_df[
                "script_reliance"
            ].mean()
        ),
    },
    "batch_alt_seed2026": {
        "original_writer_auc": float(
            original_auc_2026
        ),
        "intervened_writer_auc": float(
            intervened_auc_2026
        ),
        "residual_script_auc": float(
            residual_script_auc_2026
        ),
        "mean_script_reliance": float(
            pair_effect_2026_df[
                "script_reliance"
            ].mean()
        ),
    },
    "cross_script_reliance": {
        "mean_relative_control_to_batch_reduction": float(
            paired_reliance_df[
                "relative_reliance_reduction"
            ].mean()
        ),
        "control_mean_intervention_auc_change": float(
            paired_reliance_df[
                "control_intervention_auc_change"
            ].mean()
        ),
        "batch_alt_mean_intervention_auc_change": float(
            paired_reliance_df[
                "batch_intervention_auc_change"
            ].mean()
        ),
    },
    "random_subspace_control": {
        "seed123_control_script_to_random_reliance_ratio": float(
            script_specificity_df.loc[
                (
                    script_specificity_df[
                        "training_seed"
                    ] == 123
                )
                & (
                    script_specificity_df[
                        "model"
                    ] == "lambda0_control"
                ),
                "script_vs_random_reliance_ratio",
            ].iloc[
                0
            ]
        ),
        "seed123_batch_script_to_random_reliance_ratio": float(
            script_specificity_df.loc[
                (
                    script_specificity_df[
                        "training_seed"
                    ] == 123
                )
                & (
                    script_specificity_df[
                        "model"
                    ] == "batch_alt_lambda0p5"
                ),
                "script_vs_random_reliance_ratio",
            ].iloc[
                0
            ]
        ),
        "seed2026_control_script_to_random_reliance_ratio": float(
            script_specificity_df.loc[
                (
                    script_specificity_df[
                        "training_seed"
                    ] == 2026
                )
                & (
                    script_specificity_df[
                        "model"
                    ] == "lambda0_control"
                ),
                "script_vs_random_reliance_ratio",
            ].iloc[
                0
            ]
        ),
        "seed2026_batch_script_to_random_reliance_ratio": float(
            script_specificity_df.loc[
                (
                    script_specificity_df[
                        "training_seed"
                    ] == 2026
                )
                & (
                    script_specificity_df[
                        "model"
                    ] == "batch_alt_lambda0p5"
                ),
                "script_vs_random_reliance_ratio",
            ].iloc[
                0
            ]
        ),
    },
    "strong_uncertainty_test": {
        "seed123_score_only_auc": float(
            strong_uncertainty_123_df[
                "score_auc"
            ].mean()
        ),
        "seed123_script_auc_gain": float(
            strong_uncertainty_123_df[
                "script_auc_gain"
            ].mean()
        ),
        "seed2026_score_only_auc": float(
            strong_uncertainty_2026_df[
                "score_auc"
            ].mean()
        ),
        "seed2026_script_auc_gain": float(
            strong_uncertainty_2026_df[
                "script_auc_gain"
            ].mean()
        ),
    },
    "conclusion": {
        "supported": (
            "Script decodability and decision-level "
            "script reliance are distinct properties. "
            "Batch-level adversarial training strongly "
            "reduced cross-script decision sensitivity "
            "to the learned script-sensitive subspace "
            "without eliminating script decodability."
        ),
        "not_supported": (
            "The proposed absolute Script Reliance score "
            "did not provide complementary verification-error "
            "uncertainty after conditioning on a sufficiently "
            "strong score-based baseline."
        ),
        "interpretation_rule": (
            "Script Reliance should currently be treated "
            "as a representation and decision diagnostic, "
            "not as a validated uncertainty measure."
        ),
    },
}


summary_path = (
    REPORT_DIR
    / "notebook18_experiment_summary.json"
)

with open(
    summary_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        notebook18_summary,
        file,
        indent=2,
    )


print(
    "Summary saved:",
    summary_path.relative_to(
        PROJECT_ROOT
    ),
)

print()

print(
    "Mean control-to-batch cross-script reliance reduction (%):",
    100.0
    * paired_reliance_df[
        "relative_reliance_reduction"
    ].mean(),
)

print(
    "Seed-123 strong-baseline Script Reliance gain:",
    strong_uncertainty_123_df[
        "script_auc_gain"
    ].mean(),
)

print(
    "Seed-2026 strong-baseline Script Reliance gain:",
    strong_uncertainty_2026_df[
        "script_auc_gain"
    ].mean(),
)

Summary saved: reports/script_reliance_intervention/notebook18_experiment_summary.json

Mean control-to-batch cross-script reliance reduction (%): 72.64590822660124
Seed-123 strong-baseline Script Reliance gain: -5.279332015701854e-05
Seed-2026 strong-baseline Script Reliance gain: -3.661295969227818e-05


## Final Summary

This notebook investigated whether script information that remains decodable from writer embeddings is also used by individual writer-verification decisions.

The central distinction was:

**Script leakage is not necessarily script reliance.**

Previous experiments showed that batch-level alternating adversarial training improved cross-script writer verification without eliminating script decodability. This notebook therefore moved from representation-level leakage to decision-level intervention.

### Development-only script-subspace intervention

A script-sensitive linear subspace was estimated using development writers only.

To avoid selecting the intervention dimension on the validation set, development writers were divided into writer-disjoint subspace-training and monitoring groups. A fixed candidate grid of removal dimensions was evaluated on the development monitor writers.

Script information was highly distributed across the embedding space. Removing only a few directions had almost no effect on writer-disjoint script prediction. Even after 128 dimensions were removed, development-monitor script AUC remained approximately 0.807.

A 128-dimensional intervention was therefore selected from the predefined development-only candidate grid and subsequently kept fixed.

### Partial script suppression

For the seed-123 batch-alternating model, the independent validation script-probe AUC decreased from approximately 0.9992 before intervention to 0.8254 after the fixed 128-dimensional intervention.

For the seed-2026 batch-alternating model, residual validation script-probe AUC after the same fixed-dimensional procedure was 0.7011.

Therefore, the intervention substantially reduced linearly accessible script information, but it did not completely erase script information.

The intervention should consequently be interpreted as a **partial script-subspace intervention**, not complete script erasure.

### Decision-level script reliance

For a verification pair \(A,B\), decision-level script reliance was defined as

\[
R_{\text{script}}(A,B)
=
\left|
s(A,B)
-
s^{-script}(A,B)
\right|
\]

where \(s(A,B)\) is the original cosine similarity and \(s^{-script}(A,B)\) is the similarity after the learned script-sensitive subspace is removed.

The most important comparison was between the matched lambda-0 controls and batch-alternating adversarial models.

| Seed | Model | Original writer AUC | Intervened writer AUC | Mean cross-script reliance |
|---|---|---:|---:|---:|
| 123 | Lambda-0 control | 0.738789 | 0.768275 | 0.049595 |
| 123 | Batch-alt | 0.776425 | 0.776584 | 0.008887 |
| 2026 | Lambda-0 control | 0.738279 | 0.762910 | 0.033011 |
| 2026 | Batch-alt | 0.770031 | 0.770636 | 0.012144 |

Relative cross-script reliance decreased by approximately:

- 82.1% for seed 123
- 63.2% for seed 2026
- 72.6% on average

This reduction occurred even though script information remained decodable from the batch-alternating embeddings.

### Verification effect of the intervention

Removing the learned script-sensitive subspace improved cross-script verification in both lambda-0 control models:

- Seed 123 cross-script AUC change: +0.013647
- Seed 2026 cross-script AUC change: +0.010041
- Mean control cross-script AUC change: +0.011844

In contrast, applying the same intervention to the batch-alternating models produced almost no cross-script improvement:

- Seed 123: -0.001175
- Seed 2026: -0.001040
- Mean batch-alt change: -0.001108

This suggests that the control models contained script-sensitive components that interfered with cross-script verification, whereas the adversarially trained models were already substantially less dependent on those components.

### Random-subspace control

A matched random 128-dimensional intervention was used to test whether the observed effects were simply caused by removing a large fraction of the embedding dimensions.

For the lambda-0 controls, the learned script-sensitive subspace produced substantially larger cross-script score changes than random subspaces:

- Seed 123 script-to-random reliance ratio: 3.45
- Seed 2026 script-to-random reliance ratio: 2.40

Random-subspace removal slightly decreased cross-script AUC on average, whereas script-subspace removal improved it.

For the batch-alternating models, script-sensitive intervention effects were approximately at random-subspace scale:

- Seed 123 ratio: 0.71
- Seed 2026 ratio: 0.89

This control strengthens the interpretation that adversarial training reduced **specific decision-level dependence on script-sensitive representation dimensions**, rather than merely making the embeddings insensitive to arbitrary dimensional removal.

### Uncertainty hypothesis

An initial experiment suggested that Script Reliance could complement ordinary distance-to-threshold uncertainty for detecting cross-script verification errors.

However, a matched random-subspace sensitivity control produced a much larger apparent improvement. Further analysis showed that random sensitivity was strongly correlated with the original verification score.

A stronger score-based baseline was therefore introduced using both signed and absolute distance from the verification threshold.

With this stronger baseline:

- Seed 123 score-only error AUC: approximately 0.96556
- Seed 123 Script Reliance gain: approximately -0.00005
- Seed 2026 score-only error AUC: approximately 0.96575
- Seed 2026 Script Reliance gain: approximately -0.00004

Script Reliance also failed to improve error prediction when false-accept and false-reject decisions were analyzed separately.

Therefore, the proposed absolute Script Reliance score is **not supported as a complementary uncertainty measure** in its current form.

Earlier apparent uncertainty improvements were attributable to an insufficiently expressive score-based baseline rather than unique information provided by Script Reliance.

### Main Finding

The strongest result of this notebook is therefore not an uncertainty improvement.

Instead, the evidence supports the distinction:

\[
\text{Script Leakage}
\neq
\text{Script Reliance}
\]

Batch-level adversarial training did not eliminate script information from the representation, but it substantially reduced how strongly cross-script writer-verification decisions responded to intervention on script-sensitive representation dimensions.

In other words, a nuisance attribute can remain highly decodable from a representation while the downstream verification decision becomes substantially less dependent on that attribute.

### Supported Conclusion

**Supported:** Script decodability and decision-level script reliance are distinct properties. Across two training seeds, batch-level adversarial training substantially reduced cross-script decision sensitivity to the learned script-sensitive subspace without eliminating script decodability.

**Not supported:** The current absolute Script Reliance score should not be presented as a validated uncertainty measure.

Script Reliance should therefore currently be treated as a **representation and decision diagnostic**, not as the final uncertainty mechanism for the writer-verification system.

### Research Implication

The original uncertainty-aware writer-verification objective remains open.

The next uncertainty method should be evaluated against sufficiently strong score-based baselines from the beginning and should demonstrate complementary reliability information under writer-disjoint evaluation.

The script-reliance intervention remains valuable as a separate diagnostic contribution for distinguishing nuisance-information accessibility from actual downstream decision dependence.